# Etap III — Parametryzacja Funkcji Impedancji i WTC (Kalibracja Kosztu)

Ten notebook realizuje **Etap III** z `realization_plan.md` dla 3 miast (Dublin, Paryż, Warszawa).

Budujemy macierze czasów podróży (TTM — Travel Time Matrix) przy użyciu R5 oraz transformujemy je do macierzy kosztów uogólnionych (WTC — Whole Travel Chain Cost).

**Wejścia (kanoniczne):** wyłącznie artefakty z Etapu I (`../outputs/etap1/artifacts_index.json`).

**Wyjścia:** `../outputs/etap3/<city>/...` zgodnie z kontraktem z `realization_plan.md`.

Założenia:
- Dobieramy datę wyjazdu dynamicznie z dostępnego zakresu GTFS (preferując środek).
- Macierze TTM budujemy dla próbki jednostek OD (domyślnie k=750 origins × k=750 destinations).
- Fallback: jeśli R5 nie działa, stosujemy walk-only z ograniczeniem czasowym.


In [1]:
from __future__ import annotations

import importlib.util
import json
import os
import sys
from dataclasses import dataclass
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import geopandas as gpd


def _find_repo_root(start: Path) -> Path:
    """Find repository root by walking upwards until `realization_plan.md` is found."""
    p = start.resolve()
    for cand in [p] + list(p.parents):
        if (cand / 'realization_plan.md').exists():
            return cand
    raise FileNotFoundError(f"Cannot find repo root upwards from: {start} (resolved={p})")


try:
    _NOTEBOOK_DIR = Path(__file__).resolve().parent  # type: ignore[name-defined]
except Exception:
    _NOTEBOOK_DIR = Path.cwd()

ROOT = _find_repo_root(_NOTEBOOK_DIR)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.gtfs_utils import (
    choose_departure_datetime_within_gtfs,
    read_gtfs_service_date_range,
    choose_service_day_within_gtfs,
    gtfs_has_any_service_on_date,
)
from scripts.gtfs_merge import merge_gtfs_zips

# JVM Memory setup BEFORE r5py import
if '-Xmx10G' not in sys.argv:
    sys.argv.append('-Xmx10G')

# r5py (optional, requires Java)
HAS_R5PY = importlib.util.find_spec('r5py') is not None
if HAS_R5PY:
    import r5py

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 140)

print('Python:', sys.version.split()[0])
print('pandas:', pd.__version__)
print('numpy:', np.__version__)
print('geopandas:', gpd.__version__)
print('HAS_R5PY:', HAS_R5PY)
if HAS_R5PY:
    print('r5py:', r5py.__version__)

Python: 3.9.13
pandas: 2.3.3
numpy: 2.0.2
geopandas: 1.0.1
HAS_R5PY: True
r5py: 1.0.0


## 0) Konfiguracja (kontrakt ścieżek) + parametry analizy

### Parametry kontrolujące przebieg analizy:

**Próbkowanie jednostek OD (Origin-Destination):**
- `OD_SCALE_MODE` = `'sample_k'` - używamy stałej liczby origins i destinations (nie wszystkie komórki siatki)
- `OD_SAMPLE_K` = 750 - liczba losowo wybranych origins i destinations per miasto (750 — 750 = 562,500 par OD na miasto)
- `OD_RANDOM_SEED` = 42 - ziarno dla replikowalności losowania
- `PARIS_OD_ROLE_MODE` = `'grid_to_grid'` - dla porównywalności między miastami, wszystkie macierze TTM budujemy grid x grid (zamiast grid x employment dla Paryża)

**Dobór daty i czasu analizy:**
- `ANALYSIS_DATE_YYYYMMDD` - jeżli `None`, notebook dobiera automatycznie datę z zakresu dostępnego w GTFS (preferując środek okresu)
- `DEPARTURE_TIME_WINDOW` = `('06:00:00 - 7:30:00', '11:00:00 - 12:30:00', '19:00:00 - 20:30:00')` - analizujemy trzy okna czasowe:
  - **poranek** (rush hour): 6:00 - 7:30
  - **południe** (off-peak): 11:00 - 12:30
  - **wieczór**: 19:00 - 20:30

**Multi-departure temporal profile (opcjonalnie):**
- `USE_MULTI_DEPARTURES` = `False` (może być uaktywnione dla bardziej dokłdnych średnich czasów)
- `TEMPORAL_PROFILES` - słownik profilóww czasowych, każdy zawiera 19 godzin odljazdu w 5-minutowych interwałach
- Przydatne do uśredniania czasów podróży w zależności od headway danej linii

**Parametry spaceru (walking):**
- `WALK_SPEED_MPS` = 1.25 m/s (4.5 km/h) - typowe tempo pieszego w mieście
- `WALK_SPEED_KMH` = 4.5 km/h - przeliczenie dla wizualizacji
- `MAX_WALK_DISTANCE_M` = 2000 m - maksymalny dystans spaceru do/od przystanku (zwiększony z 1500 dla lepszego pokrycia R5)
- `MAX_TRIP_DURATION_MIN` = 180 min - maksymalny czas podróży uwzględniany w analizie

**Fallback walk-only (awaryjny spacer bez transportu):**
- `FALLBACK_WALK_ONLY_MAX_TIME_MIN` = 90.0 min - jeżli spacer trwa >90 min, oznaczamy parę OD jako nieosiągalną (obniżone z 240 min)
- `WALK_DETOUR_FACTOR` = 1.3 - korekta dystansu euklidesowego na rzeczywisty sieciowy (typowe dla miast EU: 1.2 - 1.4)

**Kontrola jakości macierzy TTM:**
- `R5_MIN_NONNULL_SHARE` = 0.01 (1%) - minimum nie-null wartości w macierzy
- `R5_MIN_NONNULL_ABS` = 500 - minimum liczby niespustych par OD (absolutnie)
- `MAX_FILL_FRACTION` = 0.95 - tolerujemy do 95% wypełnienia (brakujące wartości = nieosiągalne pary)
- `FILL_MISSING_WITH_WALK` = `True` - puste pola macierzy TTM uzupełniamy czasem spaceru (fallback)

**R5-specific parametry:**
- `R5_DEPARTURE_TIME_WINDOW_MIN` = 10 - okno randomizacji R5 w minutach (R5 próbkuje departures w ±5 min wokół‚ podanego czasu, zamiast domyślnych 60 min)

**Filtrowanie do obszaru obsługi:**
- `USE_ADMIN_BOUNDARY_FILTER` = `True` - filtrujemy OD do granic administracyjnych (predefiniowanych dla każdego miasta)
- `STOPS_PROXIMITY_BUFFER_M` = 3000 m - dodatkowo zachowujemy tylko OD w odległości <3 km od przystanku GTFS

**Mapowanie atrybutów popytowych (populacja, zatrudnienie):**
```python
CITY_POP_COLUMN = {
    'dublin': 'pop',           # Census 2022 via sjoin do gridu ITM
    'paris': 'pop',            # fra_pd_2020_1km_ASCII_XYZ.csv
    'warsaw': 'tot',           # NSP2021 census, całkowita populacja
}
CITY_JOBS_COLUMN = {
    'dublin': 'employment',    # areal-weighted z Workplace Zones
    'paris': 'employment',     # areal-weighted z danych SIRENE
    'warsaw': 'employment',    # jeżli dostępne; inaczej None
}
```
Kolumny te normalizujemy do ujednoliconych nazw `'pop'` i `'jobs'` dla obsługi w modelach grawitacyjnych.

In [2]:
os.chdir(str(ROOT))



ETAP1_DIR = ROOT / 'outputs' / 'etap1'

ARTIFACTS_ETAP1 = ETAP1_DIR / 'artifacts_index.json'

OUT_DIR = ROOT / 'outputs' / 'etap3'

OUT_DIR.mkdir(parents=True, exist_ok=True)



if not ARTIFACTS_ETAP1.exists():

    raise FileNotFoundError(f'Missing Etap1 artifacts index: {ARTIFACTS_ETAP1}')



CITIES = ['warsaw_core', 'dublin_core', 'warsaw', 'paris', 'dublin']



CITY_CRS_METRIC = {

    'dublin': 'EPSG:2157',

    'dublin_core': 'EPSG:2157',

    'paris': 'EPSG:2154',

    'warsaw': 'EPSG:2180',

    'warsaw_core': 'EPSG:2180',

}



# --- Ścieżki do granic administracyjnych (z Etapu I, cached w Data/) ---

DATA_DIR = ROOT / 'Data'

CITY_ADMIN_BOUNDARY_PATH = {

    'dublin': DATA_DIR / 'Dublin' / 'boundary_admin.geojson',

    'dublin_core': DATA_DIR / 'Dublin' / 'boundary_admin.geojson',

    'paris': DATA_DIR / 'Paris' / 'boundary_admin.geojson',

    'warsaw': DATA_DIR / 'Warsaw' / 'boundary_admin.geojson',

    'warsaw_core': DATA_DIR / 'Warsaw' / 'boundary_admin.geojson',

}



# --- Parametry OD (skalowanie) ---

OD_SCALE_MODE: str = 'sample_k'

OD_SAMPLE_K: int = 750

OD_RANDOM_SEED: int = 42

# Role OD per miasto — 'grid_to_grid' zapewnia porównywalność między miastami
PARIS_OD_ROLE_MODE: str = 'grid_to_grid'

# --- Filtrowanie OD do obszaru obsługi (admin boundary + bliskość przystanków) ---
USE_ADMIN_BOUNDARY_FILTER: bool = True

STOPS_PROXIMITY_BUFFER_M: int = 3000

# Dla *_core stosujemy ciaśniejszy bufor, aby ograniczył obszary peryferyjne i puste.

CITY_STOPS_PROXIMITY_BUFFER_M: dict = {

    'dublin_core': 800,

}

# Dodatkowe docięcie rdzenia Dublina: promieniu od centrum miasta (metryczny po reprojekcji).
DUBLIN_CORE_CENTER_WGS84: Tuple[float, float] = (-6.2603, 53.3498)
DUBLIN_CORE_MAX_RADIUS_M: int = 10000



def get_stop_buffer_m(city: str) -> int:

    return int(CITY_STOPS_PROXIMITY_BUFFER_M.get(city, STOPS_PROXIMITY_BUFFER_M))



# --- Parametry podróży (R5) ---

ANALYSIS_DATE_YYYYMMDD: Optional[str] = None

DEPARTURE_TIME_WINDOW: Tuple[str, str, str] = ('06:00:00 - 7:30:00', '11:00:00 - 12:30:00', '19:00:00 - 20:30:00')

# Multi-departure (redukcja wrażliwości na headway)
USE_MULTI_DEPARTURES: bool = False

TEMPORAL_PROFILES: dict = {

    'morning_peak': ('06:00:00', '06:05:00', '06:10:00', '06:15:00', '06:20:00', '06:25:00', '06:30:00', '06:35:00', '06:40:00', '06:45:00', '06:50:00', '06:55:00', '07:00:00', '07:05:00', '07:10:00', '07:15:00', '07:20:00', '07:25:00', '07:30:00'),

    'midday_offpeak': ('11:00:00', '11:05:00', '11:10:00', '11:15:00', '11:20:00', '11:25:00', '11:30:00', '11:35:00', '11:40:00', '11:45:00', '11:50:00', '11:55:00', '12:00:00', '12:05:00', '12:10:00', '12:15:00', '12:20:00', '12:25:00', '12:30:00'),

    'evening': ('19:00:00', '19:05:00', '19:10:00', '19:15:00', '19:20:00', '19:25:00', '19:30:00', '19:35:00', '19:40:00', '19:45:00', '19:50:00', '19:55:00', '20:00:00', '20:05:00', '20:10:00', '20:15:00', '20:20:00', '20:25:00', '20:30:00'),

}

MULTI_DEPARTURE_AGG: str = 'median'



# Walk / transit parametry

WALK_SPEED_MPS: float = 1.25

WALK_SPEED_KMH: float = WALK_SPEED_MPS * 3.6
MAX_WALK_DISTANCE_M: int = 2000  # zwiększone z 1500 dla lepszego pokrycia R5
MAX_TRIP_DURATION_MIN: int = 180



# Parametry fallback (walk-only)
FALLBACK_WALK_ONLY_MAX_TIME_MIN: float = 90.0  # obniżone z 240 — >90 min marszu ≈ nieosiągalne
WALK_DETOUR_FACTOR: float = 1.3  # korekta dystansu euklidesowego → sieciowy (1.2–1.4 typowe w miastach EU)

# QC thresholds

R5_MIN_NONNULL_SHARE: float = 0.01

R5_MIN_NONNULL_ABS: int = 500

# Fill / sparsity — zwiększono MAX_FILL_FRACTION by akceptować fill zamiast pustej macierzy
MAX_FILL_FRACTION: float = 0.95

FILL_MISSING_WITH_WALK: bool = True

DROP_UNSNAPPED_POINTS: bool = False

# R5 departure_time_window: czas (w minutach) wewnętrznej randomizacji R5
# Ustawiony na 10 min, żeby R5 próbkował departures w krótkim oknie wokół
# jawnie podanych godzin (zamiast domyślnych 60 min, które rozszerzały okno do 09:30).
R5_DEPARTURE_TIME_WINDOW_MIN: int = 10

# Snap-to-streets params — obniżone progi for better R5 coverage
SNAP_TO_STREETS_BEFORE_R5: bool = False

SNAP_MAX_DIST_M: float = 2000.0
SNAP_MIN_KEEP_SHARE: float = 0.50

# Diagnostic params
DIAG_DESTINATIONS_FROM_ORIGINS: bool = False
DIAG_SAMPLE_K: int = 200

# --- Mapowanie kolumn atrybutowych per miasto (populacja / zatrudnienie) ---
# Używane do ekstrakcji O_i (popyt) i D_j (podaż) z warstw Etapu I
CITY_POP_COLUMN = {
    'dublin': 'pop',      # z Census 2022 GISCO grid → sjoin do ITM grid (RC3, etap1)
    'paris': 'pop',       # z fra_pd_2020_1km_ASCII_XYZ.csv → cell_id, pop
    'warsaw': 'tot',      # z NSP2021 census → tot = populacja ogółem
}
CITY_JOBS_COLUMN = {
    'dublin': 'employment',   # areal-weighted z Workplace Zones → kolumna 'employment' (Etap I)
    'paris': 'employment',    # areal-weighted z workplace_density (SIRENE) → kolumna 'employment' (Etap I)
    'warsaw': 'employment',
}

print('ROOT:', ROOT)
print('OUT_DIR:', OUT_DIR)
print('OD_SCALE_MODE:', OD_SCALE_MODE, 'k=', OD_SAMPLE_K, 'seed=', OD_RANDOM_SEED)
print('PARIS_OD_ROLE_MODE:', PARIS_OD_ROLE_MODE)
print('USE_ADMIN_BOUNDARY_FILTER:', USE_ADMIN_BOUNDARY_FILTER, 'stop_buffer_m=', STOPS_PROXIMITY_BUFFER_M)
print('R5_DEPARTURE_TIME_WINDOW_MIN:', R5_DEPARTURE_TIME_WINDOW_MIN)
print('HAS_R5PY:', HAS_R5PY)

ROOT: C:\Users\Michc\Dropbox\IO_UW\Magisterka\Masters
OUT_DIR: C:\Users\Michc\Dropbox\IO_UW\Magisterka\Masters\outputs\etap3
OD_SCALE_MODE: sample_k k= 750 seed= 42
PARIS_OD_ROLE_MODE: grid_to_grid
USE_ADMIN_BOUNDARY_FILTER: True stop_buffer_m= 3000
R5_DEPARTURE_TIME_WINDOW_MIN: 10
HAS_R5PY: True


In [3]:
artifacts_etap1 = json.loads(ARTIFACTS_ETAP1.read_text(encoding='utf-8'))


## 1) Funkcje pomocnicze (ładowanie danych, mapowanie ścieżek)

### Cel i implementacja:

Ta sekcja zawiera utility functions niezędne do całego pipeline'u:

**`resolve_artifact_path(p: str) -> Path`** - mapowanie ścieżek artefaktów z Etapu I
- Funkcja obsługuje różne formaty ścieżek (absolutne, względne, z Dropboxa)
- Jeśli ścieżka zawiera marker `/outputs/etap1/`, wydobywamy sufiks i mapujemy na lokalny `ROOT/outputs/etap1/suffix`
- Fallback: jeżli plik nie istnieje, próbujemy `ROOT / p` (ścieżka względna od roota projektu)
- Ratuje nas przed błędami ścieżek, gdy repository jest klonowany w innym miejscu lub przenoszony między maszynami

**`CityInputs` dataclass** - enkapsulacja wejścia dla miasta
- `city` - nazwa miasta (np. `'warsaw'`, `'paris_core'`)
- `gtfs_tables` - słownik mapujący nazwy tabel (stops, routes, trips, stop_times) na `Path` do CSV
- `spatial_layers` - słownik mapujący nazwy warstw ([pop_grid_1km], [employment_zones]) na `Path` do GeoJSON

**`build_inputs(city: str) -> CityInputs`** - budowanie struktury wejściowej dla miasta
- Waliduje, że wszystkie wymagane tabele GTFS istnieją w `artifacts_etap1.get('gtfs')[city]['tables']`
- Mapuje ścieżki do warstw przestrzennych (jeżli istnieją 'metric_geojson' dla danej warstwy)
- Rzuca `KeyError`, jeśli brakują krytyczne tabele (stops, routes, trips, stop_times)
- Pozwala na "soft fail" dla opcjonalnych warstw (np. employment zones w Warszawie)

**`_rel(p) -> str`** - konwersja ścieżki na względną względem ROOT
- Ułatwia odczytywanie logów i raportów (bez pełnych ścieżek absolutnych)
- Fallback na ścieżkę absolutną, jeśli relatywizacja nie jest możliwa

In [4]:
def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat(timespec='seconds').replace('+00:00', 'Z')


def resolve_artifact_path(p: str) -> Path:
    """Mapuje ścieżkę z artifacts_index.json na istniejący plik lokalny.

    Etap1 może mieć ścieżki absolutne z innego środowiska (np. Dropbox).
    Jeżeli plik nie istnieje, próbujemy podmienić prefiks na lokalny ROOT.
    """
    path = Path(p)
    if path.exists():
        return path

    s = str(p).replace('\\', '/')
    marker = '/outputs/etap1/'
    if marker in s:
        suffix = s.split(marker, 1)[1]
        candidate = ROOT / 'outputs' / 'etap1' / suffix
        if candidate.exists():
            return candidate

    candidate = (ROOT / p).resolve()
    if candidate.exists():
        return candidate

    raise FileNotFoundError(f'Cannot resolve artifact path: {p}')


@dataclass(frozen=True)
class CityInputs:
    city: str
    gtfs_tables: Dict[str, Path]
    spatial_layers: Dict[str, Path]


REQ_GTFS_TABLES = ['stops', 'routes', 'trips', 'stop_times']


def build_inputs(city: str) -> CityInputs:
    gtfs_tables_raw = artifacts_etap1.get('gtfs', {}).get(city, {}).get('tables', {})
    spatial_layers_raw = artifacts_etap1.get('spatial', {}).get(city, {}).get('layers', {})

    if not gtfs_tables_raw:
        raise KeyError(f'No GTFS tables listed for city={city} in artifacts_index.json')

    for t in REQ_GTFS_TABLES:
        if t not in gtfs_tables_raw:
            raise KeyError(f'{city}: missing required GTFS table in Etap1 artifacts: {t}')

    gtfs_tables: Dict[str, Path] = {}
    for k, p in gtfs_tables_raw.items():
        try:
            gtfs_tables[k] = resolve_artifact_path(p)
        except FileNotFoundError:
            if k in REQ_GTFS_TABLES:
                raise

    spatial_layers: Dict[str, Path] = {}
    for layer_name, layer_paths in spatial_layers_raw.items():
        if isinstance(layer_paths, dict) and 'metric_geojson' in layer_paths:
            spatial_layers[layer_name] = resolve_artifact_path(layer_paths['metric_geojson'])

    return CityInputs(city=city, gtfs_tables=gtfs_tables, spatial_layers=spatial_layers)


inputs_by_city: Dict[str, CityInputs] = {c: build_inputs(c.replace('_core','')) for c in CITIES}
inputs_by_city



def _rel(p) -> str:
    """Konwertuje ścieżkę na względną względem ROOT (Fix 3.6)."""
    try:
        return str(Path(p).relative_to(ROOT))
    except ValueError:
        return str(p)

## 2) Global run_info + inventory (w stylu etap1/etap2)

### Powód:
Replikowalność i audytowalność wyników wymagają precyzyjnych informacji o warunkach, w jakich wygenerowaliśmy TTM.

### Co zapisujemy do `run_info.json`:
- **Timestamp UTC** - dokładny czas uruchomienia notebooka
- **Parametry analizy OD:**
  - `od_scale_mode`, `od_sample_k`, `od_random_seed` - warunki próbkowania
  - `paris_od_role_mode` - czy Paris to grid x grid czy grid x employment
  - `analysis_date_yyyymmdd` - konkretna data GTFS używana
  - `departure_time_window`, `use_multi_departures` - warunki czasowe
- **Parametry TZ:**
  - Prędkość spaceru, maksymalne dystanse, fallback walk-only
  - Progi jakości QC (R5_MIN_NONNULL_SHARE, MAX_FILL_FRACTION)
- **Wersje bibliotek** - Python, Pandas, NumPy, GeoPandas, dostępność R5py
- **Lista miast** - które miasta analizowaliśmy

### Co zapisujemy do `inventory.csv`:
Tabelę wszystkich wejściowych artefaktów per miasto:
- **Type** - 'gtfs_table_etap1' (np. stops, routes), 'spatial_layer_metric_etap1' (grid_1km, employment), 'admin_boundary'
- **City** - kod miasta
- **Name** - konkretna nazwa warstwy lub tabeli
- **Path** - ścieżka do pliku
- **Exists** - boolean czy plik faktycznie istnieje (ułatwia diagnostykę przy brakujących danych)

To umożliwia szybkie sprawdzenie, czy wszystkie wejścia sś dostępne, zanim zaczniemy obliczenia TTM (które mogą trwać godziny).

In [5]:
run_info = {
    'run_utc': utc_now_iso(),
    'root': str(ROOT),
    'out_dir': str(OUT_DIR),
    'inputs': {
        'etap1_artifacts_index': str(ARTIFACTS_ETAP1),
    },
    'analysis': {
        'od_scale_mode': OD_SCALE_MODE,
        'od_sample_k': OD_SAMPLE_K,
        'od_random_seed': OD_RANDOM_SEED,
        'paris_od_role_mode': PARIS_OD_ROLE_MODE,
        'analysis_date_yyyymmdd': ANALYSIS_DATE_YYYYMMDD,
        'departure_time_window': DEPARTURE_TIME_WINDOW,
        'use_admin_boundary_filter': USE_ADMIN_BOUNDARY_FILTER,
        'stops_proximity_buffer_m': STOPS_PROXIMITY_BUFFER_M,
        'city_stops_proximity_buffer_m': CITY_STOPS_PROXIMITY_BUFFER_M,
        'walk_speed_mps': WALK_SPEED_MPS,
        'walk_detour_factor': WALK_DETOUR_FACTOR,
        'max_walk_distance_m': MAX_WALK_DISTANCE_M,
        'max_trip_duration_min': MAX_TRIP_DURATION_MIN,
        'fallback_walk_only_max_time_min': FALLBACK_WALK_ONLY_MAX_TIME_MIN,
        'max_fill_fraction': MAX_FILL_FRACTION,
    },
    'versions': {
        'python': sys.version.split()[0],
        'pandas': pd.__version__,
        'numpy': np.__version__,
        'geopandas': gpd.__version__,
        'r5py_available': HAS_R5PY,
    },
    'cities': CITIES,
}

(OUT_DIR / 'run_info.json').write_text(json.dumps(run_info, ensure_ascii=False, indent=2), encoding='utf-8')

inv_rows = []
for city, cin in inputs_by_city.items():
    for name, path in sorted(cin.gtfs_tables.items()):
        inv_rows.append({'city': city, 'type': 'gtfs_table_etap1', 'name': name, 'path': str(path), 'exists': path.exists()})
    for name, path in sorted(cin.spatial_layers.items()):
        inv_rows.append({'city': city, 'type': 'spatial_layer_metric_etap1', 'name': name, 'path': str(path), 'exists': path.exists()})
    # Dodaj boundary do inwentarza
    bp = CITY_ADMIN_BOUNDARY_PATH.get(city)
    if bp:
        inv_rows.append({'city': city, 'type': 'admin_boundary', 'name': 'boundary_admin', 'path': str(bp), 'exists': bp.exists()})

inventory_df = pd.DataFrame(inv_rows)
inventory_df.to_csv(OUT_DIR / 'inventory.csv', index=False)
inventory_df.head(15)

,city,type,name,path,exists
0,warsaw_core,gtfs_table_etap1,agency,outputs\etap1\warsaw\gtfs_normalized\agency.csv,True
1,warsaw_core,gtfs_table_etap1,attributions,outputs\etap1\warsaw\gtfs_normalized\attributi...,True
2,warsaw_core,gtfs_table_etap1,calendar_dates,outputs\etap1\warsaw\gtfs_normalized\calendar_...,True
3,warsaw_core,gtfs_table_etap1,fare_attributes,outputs\etap1\warsaw\gtfs_normalized\fare_attr...,True
4,warsaw_core,gtfs_table_etap1,fare_rules,outputs\etap1\warsaw\gtfs_normalized\fare_rule...,True
5,warsaw_core,gtfs_table_etap1,feed_info,outputs\etap1\warsaw\gtfs_normalized\feed_info...,True
6,warsaw_core,gtfs_table_etap1,frequencies,outputs\etap1\warsaw\gtfs_normalized\frequenci...,True
7,warsaw_core,gtfs_table_etap1,routes,outputs\etap1\warsaw\gtfs_normalized\routes.csv,True
8,warsaw_core,gtfs_table_etap1,shapes,outputs\etap1\warsaw\gtfs_normalized\shapes.csv,True
9,warsaw_core,gtfs_table_etap1,stop_times,outputs\etap1\warsaw\gtfs_normalized\stop_time...,True


## 3) Funkcje walidacji QC (TTM - Travel Time Matrix)

### Problemy, które chcemy wyłapać:

Macierze z R5 mogą zawierać artefakty lub błędy:
- **R5 crashed** - alimentacyjny bład, całkowicie pusta macierz
- **Constant travel times** - R5 zwrócił‚ jedną sztywną wartość (np. `180` min dla wszystkich) zamiast rzeczywistych czasów
- **Sparse matrix** - zbyt mało niepustych par OD (poniżej progu)

### Implementacja walidatorów:

**`_is_suspicious_constant_times(ttm, round_to=3, top1_share_threshold=0.98, debug=False) -> bool`**
- Heurystyka: zwraca `True` jeśli >98% wartości to jedna liczba (zaokrąglona do 3 miejsc)
- Dodatkowo filtruje parę OD do tych w obrębie `STOPS_PROXIMITY_BUFFER_M` od przystanków (dla statystyki)
- Debug: wypisuje top value i share dla diagnostyki
- Chwyci przypadki jak `travel_time_min = 180` dla wszystkich par

**`_ensure_crs(gdf, crs) -> gdf`**
- Waliduje, że GeoDataFrame ma CRS ustawiony
- Jeśli CRS nie zgadza się, reprojektuje do docelowego
- Zapobiega niewidocznym błędom przestrzennym (np. obliczenia odległości w stopniach zamiast metrów)

**`_stable_unit_id_from_centroid_xy(x, y) -> str`**
- Generuje stabilny, czytelny ID dla komórki siatki na podstawie współrzędnych centroidu
- Format: `cell_x.xxx_y.yyy` (zaokrąglone do mm w lokalnym CRS metrycznym)
- Gwarancja: ID błędzie ten sam dla tego samego centroidu, niezależnie od runów

Te walidatory są wywoływane w wypadku każdego miasta, aby zidentyfikować problemy przed zapisaniem TTM.

In [6]:
def _is_suspicious_constant_times(ttm: pd.DataFrame, round_to: int = 3, top1_share_threshold: float = 0.98, debug: bool = False) -> bool:
    """Heuristic: True if almost all non-null travel times are identical.

    This catches the failure mode where r5 returns a constant capped value (or a single travel time) for most OD pairs.
    """
    if ttm is None or 'travel_time_min' not in ttm.columns:
        return True
    # --- DESCRIPTIVE STATS: Calculate only using OD pairs within 3 km of stops ---
    city_buffer_m = get_stop_buffer_m(city)
    if ttm is not None and city_buffer_m > 0:
        print(f'\n>> QC Stats: Filtering to O-D pairs within {city_buffer_m}m of gtfs stops...')
        # filter requires 'unit_id' column
        o_temp = origins.rename(columns={'origin_id':'unit_id'})
        d_temp = destinations.rename(columns={'dest_id':'unit_id'})
        _ou, _ = filter_units_to_service_area(o_temp, city, buffer_m=city_buffer_m)
        _du, _ = filter_units_to_service_area(d_temp, city, buffer_m=city_buffer_m)
        valid_o = _ou['unit_id'].unique()
        valid_d = _du['unit_id'].unique()

        mask = ttm['origin_id'].isin(valid_o) & ttm['dest_id'].isin(valid_d)
        ttm_qc = ttm.loc[mask].copy()
        print(f'>> QC Stats: {len(ttm_qc)} valid pairs out of {len(ttm)} total for {city}\n')
    else:
        ttm_qc = ttm if ttm is not None else None

    s = pd.to_numeric(ttm_qc['travel_time_min'], errors='coerce') if ttm_qc is not None else pd.Series(dtype=float)
    if s.empty:
        return True
    vc = s.round(round_to).value_counts(normalize=True)
    top1_share = float(vc.iloc[0])
    if debug:
        print(f'  QC constant check: top1_share={top1_share:.3f}, threshold={top1_share_threshold}, top_value={vc.index[0]}')
    return top1_share >= float(top1_share_threshold)


def _ensure_crs(gdf: gpd.GeoDataFrame, crs: str) -> gpd.GeoDataFrame:
    if gdf.crs is None:
        raise ValueError('GeoDataFrame has no CRS set')
    if str(gdf.crs) != crs:
        return gdf.to_crs(crs)
    return gdf


def _stable_unit_id_from_centroid_xy(x: float, y: float) -> str:
    # stabilny, czytelny ID (zaokrąglenie do mm w CRS metrycznym)
    return f'cell_{x:.3f}_{y:.3f}'

In [7]:
# --- Filtrowanie jednostek OD do obszaru obsługi transportu zbiorowego ---

def load_admin_boundary(city: str) -> gpd.GeoDataFrame:
    """Wczytaj granicę administracyjną z Etapu I (WGS84 → metryczny CRS miasta).
    Dodano: możliwość wycięcia tylko miast rdzeniowych (core) bez obszarów obwarzankowych.
    """
    # --- CORE CITIES FILTERING (WWA & DUB) ---
    if city == 'warsaw_core':
        path = CITY_ADMIN_BOUNDARY_PATH.get('warsaw')
        gdf = gpd.read_file(path)
        gdf = gdf[gdf['name'] == 'Warsaw']
        if gdf.crs is None: gdf = gdf.set_crs('EPSG:4326')
        return gdf.to_crs(CITY_CRS_METRIC['warsaw'])
    elif city == 'dublin_core':
        path = CITY_ADMIN_BOUNDARY_PATH.get('dublin')
        gdf = gpd.read_file(path)
        gdf = gdf[gdf['name'] == 'County Dublin']
        if gdf.crs is None: gdf = gdf.set_crs('EPSG:4326')
        return gdf.to_crs(CITY_CRS_METRIC['dublin'])
    city_key = city.replace('_core', '')
    path = CITY_ADMIN_BOUNDARY_PATH.get(city_key)
    if not path or not path.exists():
        raise FileNotFoundError(f'Admin boundary not found for {city}: {path}')
    gdf = gpd.read_file(path)
    if gdf.crs is None:
        gdf = gdf.set_crs('EPSG:4326')
    return gdf.to_crs(CITY_CRS_METRIC[city_key])


def load_gtfs_stops_as_points(city: str) -> gpd.GeoDataFrame:
    """Wczytaj przystanki z Etapu I jako punkty GeoDataFrame w metrycznym CRS."""
    cin = inputs_by_city[city]
    stops_path = cin.gtfs_tables.get('stops')
    if not stops_path or not stops_path.exists():
        raise FileNotFoundError(f'Stops CSV not found for {city}')
    stops = pd.read_csv(stops_path, dtype='string')
    stops['stop_lat'] = pd.to_numeric(stops['stop_lat'], errors='coerce')
    stops['stop_lon'] = pd.to_numeric(stops['stop_lon'], errors='coerce')
    stops = stops.dropna(subset=['stop_lat', 'stop_lon'])
    gdf = gpd.GeoDataFrame(
        stops,
        geometry=gpd.points_from_xy(stops['stop_lon'], stops['stop_lat']),
        crs='EPSG:4326',
    )
    city_key = city.replace('_core', '')
    gdf = _ensure_crs(gdf, CITY_CRS_METRIC[city_key])
    return gdf


def filter_units_to_service_area(
    units_metric: gpd.GeoDataFrame,
    city: str,
    buffer_m: int = STOPS_PROXIMITY_BUFFER_M,
) -> Tuple[gpd.GeoDataFrame, dict]:
    """Filtruj jednostki OD do granicy administracyjnej + bliskości przystanków.

    Krok 1: Zachowaj jednostki, których centroid leży w obrębie granicy administracyjnej.
    Krok 2: Zachowaj jednostki, których centroid jest w zasięgu buffer_m od przystanku GTFS.

    Returns: (filtered_units, qc_dict)
    """
    if units_metric is None or len(units_metric) == 0:
        return units_metric, {'n_input': 0, 'n_after_admin_boundary': 0, 'n_after_stop_proximity': 0}

    n_input = len(units_metric)

    # Krok 1: granica administracyjna
    boundary = load_admin_boundary(city)
    boundary_union = boundary.union_all()

    cents = units_metric.geometry.centroid
    mask_boundary = cents.within(boundary_union)
    units_in_boundary = units_metric.loc[mask_boundary.to_numpy()].copy()
    n_after_boundary = len(units_in_boundary)

    if n_after_boundary == 0:
        print(f'  {city}: WARNING — 0 units within admin boundary! Skipping stop filter.')
        qc = {'n_input': n_input, 'n_after_admin_boundary': 0, 'n_after_stop_proximity': 0, 'kept_share': 0.0}
        return gpd.GeoDataFrame(units_in_boundary, geometry='geometry', crs=units_metric.crs), qc

    # Krok 2: bliskość przystanków (sjoin_nearest z max_distance)
    stops = load_gtfs_stops_as_points(city)

    cents_gdf = gpd.GeoDataFrame(
        {'_orig_idx': units_in_boundary.index},
        geometry=units_in_boundary.geometry.centroid.values,
        crs=units_metric.crs,
    )
    if buffer_m <= 0:
        print(f'  {city}: Skipping stop proximity filter (buffer_m={buffer_m})')
        qc = {'n_input': n_input, 'n_after_admin_boundary': n_after_boundary, 'n_after_stop_proximity': n_after_boundary, 'stop_buffer_m': buffer_m, 'kept_share': float(n_after_boundary/n_input)}
        return units_in_boundary, qc

    nearest = gpd.sjoin_nearest(
        cents_gdf, stops[['geometry']].copy(), how='left',
        max_distance=float(buffer_m), distance_col='_dist',
    )
    keep_idx = nearest.dropna(subset=['_dist'])['_orig_idx'].unique()
    units_near_stops = units_in_boundary.loc[units_in_boundary.index.isin(keep_idx)].copy()
    n_after_stops = len(units_near_stops)

    qc = {
        'n_input': n_input,
        'n_after_admin_boundary': n_after_boundary,
        'n_after_stop_proximity': n_after_stops,
        'stop_buffer_m': buffer_m,
        'n_stops_loaded': len(stops),
        'kept_share': float(n_after_stops / n_input) if n_input else 0.0,
    }
    print(f'  {city}: service area filter: {n_input} -> {n_after_boundary} (boundary) -> {n_after_stops} (stops <={buffer_m}m)')
    return gpd.GeoDataFrame(units_near_stops, geometry='geometry', crs=units_metric.crs), qc

print('Service area filter functions defined.')

Service area filter functions defined.


## 4) Funkcje ładowania jednostek OD i próbkowania

**`load_administrative_boundary(city: str) -> gpd.GeoDataFrame`**
- Wczytuje predefiniowane granice administracyjne z Etapu I (GeoJSON w WGS84)
- Reprojektuje do lokalnego CRS metrycznego miasta (EPSG:2157 Dublin, EPSG:2154 Paryż, EPSG:2180 Warszawa)
- Obsługuje "_core" warianty miast (np. `warsaw_core` → filtruje do samej Warszawy, pomijaijąc powiaty)
  - `warsaw_core`: tylko samą Warszawę powiat warszawski
  - `dublin_core`: tylko County Dublin
- Rzuca `FileNotFoundError` jeśli granica nie istnieje (krityczne dla filtracji OD)

**`load_gtfs_stops_as_points(city: str) -> gpd.GeoDataFrame`**
- Wczytuje tabelę GTFS `stops.csv` z Etapu I jako GeoDataFrame
- Konwertuje kolumny `stop_lat`, `stop_lon` (które mogą być string'ami) na float
- Filtruje puste współrzędne
- Buduje geometrię punktów (WGS84 → lokalny CRS metryczny)
- Zwraca GeoDataFrame w metrycznym CRS (gotowy do operacji `sjoin_nearest`)

**`filter_units_to_service_area(units_metric, city, buffer_m=3000) -> (gdf, qc_dict)`**
- **Krok 1:** Filtracja do granic administracyjnych
  - Oblicza centroid każdej komórki siatki (`geometry.centroid`)
  - Sprawdza, czy centroid leży wewnątrz `boundary.union_all()`
  - QC: zwraca liczbę zachowanych komórek (`n_after_admin_boundary`)

- **Krok 2:** Filtracja do bliskości przystanków
  - Dla każdego centroidu, Find nearest stop przy użyciu `sjoin_nearest(..., max_distance=buffer_m)`
  - Zachowuje komórki z przystankiem w odległości â‰¤ `buffer_m` (domyślnie 3000 m)
  - QC: zlicza liczbę przystanków załadowanych, dystans medianowy itp.

- **Wyjście QC:**
  ```python
  {
    'n_input': liczba oryginalnych komórek,
    'n_after_admin_boundary': po filtyacji do granic,
    'n_after_stop_proximity': po filtracji do bliskości przystanków,
    'kept_share': (n_after / n_input),
    'n_stops_loaded': liczba przystanków czytanych GDAFrame,
    'stop_buffer_m': wartość bufora (dla audytu),
  }
  ```

### Budowanie i próbkowanie OD:

**`load_od_units(city, role) -> gpd.GeoDataFrame`**
- Ładuje warstwy OD z Etapu I jako poligony (komórki siatki lub strefy pracy)
- Rola:
  - `'grid'` - siatka 1 km (wspólna dla wszystkich miast)
  - `'employment'` - strefy pracy (dostępne dla Dublin, Paryż; opcjonalne dla Warszawy)
- **Normalizacja atrybutów:**
  - Kolumna populacji → ujednolicona nazwa `'pop'` (czyta z `CITY_POP_COLUMN`)
  - Kolumna zatrudnienia → ujednolicona nazwa `'jobs'` (czyta z `CITY_JOBS_COLUMN`)
  - Fallback: jeśli `'pop'` brakuje ale istnieje `'Z'` (Paryż), używamy `'Z'`
- Filtruje puste geometrie

**`build_centroids_with_ids(units, id_col=None) -> gpd.GeoDataFrame`**
- Konwertuje poligony na centroidy
- Dodaje kolumny `'x'`, `'y'` (współrzędne centroidu)
- Generuje mądre ID:
  - Jeśli `id_col` istnieje, używamy tej kolumny
  - InWYNaczęj: ID = `_stable_unit_id_from_centroid_xy(x, y)` (format: `cell_x.xxx_y.yyy`)
  - Gwarancja unikalności: jeśli są duplikaty, rozróżniamy je sufixem `_0`, `_1`...

**`sample_units(gdf, k, seed) -> gdf`**
- Jeśli `len(gdf) â‰¤ k`, zwracamy wszystko
- Inaczej: losowy sample `k` wierszy z seed'em (`random_state`)
- Zapewnia replikowalność między runami

In [8]:
def load_od_units(city: str, role: str) -> gpd.GeoDataFrame:
    """Ładuje jednostki OD jako geometrię poligonów (z Etapu I).

    role:
    - 'grid' -> grid 1km (Dublin/Warsaw) lub 'pop_grid_1km' dla Paris
    - 'employment' -> employment zones (Paris) / work_zones (Dublin)
    """
    cin = inputs_by_city[city]
    city_key = city.replace('_core', '')
    if city_key == 'dublin':
        if role == 'grid':
            layer_name = 'grid_1km'
        elif role == 'employment':
            layer_name = 'work_zones'
        else:
            raise ValueError(f'dublin: role must be grid or employment, got {role}')
    elif city_key == 'warsaw':
        if role != 'grid':
            raise ValueError(f'{city}: only role=grid supported')
        layer_name = 'grid_1km'
    elif city_key == 'paris':
        if role == 'grid':
            layer_name = 'pop_grid_1km'
        elif role == 'employment':
            layer_name = 'employment_zones'
        else:
            raise ValueError('paris: role must be grid or employment')
    else:
        raise ValueError('Unknown city')

    if layer_name not in cin.spatial_layers:
        raise KeyError(f'{city}: missing spatial layer in Etap1 artifacts: {layer_name}')

    gdf = gpd.read_file(cin.spatial_layers[layer_name])
    gdf = _ensure_crs(gdf, CITY_CRS_METRIC[city_key])
    gdf = gdf[~gdf.geometry.is_empty & gdf.geometry.notna()].copy()

    # --- Normalizacja atrybutów popytowych → ujednolicone nazwy 'pop' i 'jobs' ---
    pop_col = CITY_POP_COLUMN.get(city_key)
    jobs_col = CITY_JOBS_COLUMN.get(city_key)

    if pop_col and pop_col in gdf.columns and 'pop' not in gdf.columns:
        gdf['pop'] = pd.to_numeric(gdf[pop_col], errors='coerce')
    elif pop_col and pop_col in gdf.columns:
        gdf['pop'] = pd.to_numeric(gdf['pop'], errors='coerce')

    if jobs_col and jobs_col in gdf.columns and 'jobs' not in gdf.columns:
        gdf['jobs'] = pd.to_numeric(gdf[jobs_col], errors='coerce')

    # Paris pop_grid: kolumna 'Z' → 'pop' (fallback)
    if city_key == 'paris' and role == 'grid' and 'pop' not in gdf.columns and 'Z' in gdf.columns:
        gdf['pop'] = pd.to_numeric(gdf['Z'], errors='coerce')

    return gdf


def build_centroids_with_ids(units: gpd.GeoDataFrame, id_col: Optional[str] = None) -> gpd.GeoDataFrame:
    c = units.copy()
    c['geometry'] = c.geometry.centroid
    c['x'] = c.geometry.x
    c['y'] = c.geometry.y

    if id_col and id_col in c.columns:
        c['unit_id'] = c[id_col].astype('string')
    else:
        c['unit_id'] = [
            _stable_unit_id_from_centroid_xy(float(x), float(y))
            for x, y in zip(c['x'].to_numpy(), c['y'].to_numpy())
        ]

    # Zagwarantuj unikalność
    if c['unit_id'].duplicated().any():
        dup = c['unit_id'].duplicated(keep=False)
        c.loc[dup, 'unit_id'] = c.loc[dup, 'unit_id'] + '_' + c.loc[dup].groupby('unit_id').cumcount().astype(str)

    return c[['unit_id', 'x', 'y', 'geometry'] + [col for col in c.columns if col not in {'unit_id','x','y','geometry'}]]


def sample_units(gdf: gpd.GeoDataFrame, k: int, seed: int) -> gpd.GeoDataFrame:
    if len(gdf) <= k:
        return gdf
    return gdf.sample(n=k, random_state=seed).copy()


def write_city_run_info(
    city: str,
    city_dir: Path,
    engine_mode: str,
    notes: Optional[str] = None,
    departure_datetimes: Optional[List[str]] = None,
) -> None:
    info = {
        'run_utc': utc_now_iso(),
        'root': str(ROOT),
        'out_dir': str(OUT_DIR),
        'city': city,
        'analysis': run_info['analysis'],
        'engine_mode': engine_mode,
        'departure_datetimes_used': departure_datetimes,
        'notes': notes,
        'versions': run_info['versions'],
    }
    (city_dir / 'run_info.json').write_text(json.dumps(info, ensure_ascii=False, indent=2), encoding='utf-8')

## 5) Budowa jednostek OD (origins/destinations) per miasto

### Logika per miasto:

**Dublin & Warsaw & Paris:**
- `role='grid'` + `role='grid'` - **grid-to-grid** (grid jako sources i destinations)
- Próbka: `sample_units(k=750)` losowo wybranych origins i 750 losowo wybranych destinations
- Filtracja do granic administracyjnych + `STOPS_PROXIMITY_BUFFER_M` (3 km od przystanków)
- Normalizacja atrybutów: kolumny populacji - `'pop'`, zatrudnienia - `'jobs'`

**Paryż (alternatywnie, jeśli będzie potrzebne):**
- Jeśli `PARIS_OD_ROLE_MODE = 'employment'`: grid x employment_zones (sources = grid, destinations = strefy pracy)
- Aktualnie: używamy `'grid_to_grid'` dla porównywalności

### Operacje na każde miasto:

1. **Załaduj layer:**
   - `load_od_units(city, role='grid')` → warstwa OD jako poligony
   - Normalizacja CRS do lokalnego (metrycznego) miasta

2. **Filtruj do obszaru obsługi:**
   - `filter_units_to_service_area(units, city, buffer_m=3000)`
   - Zachowujemy tylko komórki w obrębie granic + 3 km od przystanków
   - QC logging: ile wypadło po każdym kroku

3. **Zbuduj centroidy with IDs:**
   - `build_centroids_with_ids(units_filtered)`
   - Centroidy z stabilnymi ID (`cell_x.xxx_y.yyy`)
   - Zwracamy GeoDataFrame z kolumnami: `unit_id`, `x`, `y`, `geometry`, `pop`, `jobs`, itp.

4. **Próbkuj origins i destinations:**
   - `sample_units(origins_gdf, k=750, seed=42)` → losowe 750 origins
   - `sample_units(destinations_gdf, k=750, seed=42+1)` → losowe 750 destinations (inne ziarno)
   - **Uwaga:** jeśli te same jednostki → origins == destinations (autoreferencje są ok, będą 0 minut)

5. **Zapisz GeoJSON + CSV:**
   - `outputs/etap3/{city}/od_units/origins.geojson` - origins z atrybutami (geometry + pop + jobs)
   - `outputs/etap3/{city}/od_units/destinations.geojson` - destinations
   - `outputs/etap3/{city}/od_units/origins_lookup.csv` - szybka tablica lookupowa (unit_id, x, y)
   - `outputs/etap3/{city}/od_units/destinations_lookup.csv` - analogicznie

### Liczby i logistyka:

- **Per miasto:** 750 origins x 750 destinations = 562,500 par OD
- **Razem 3 miasta:** ~1.7 mln par OD
- **Przesyłanie do R5:** każde 750 par jest dzielone w buczach (jeśli to potrzebne dla wydajności)
- **Storage:** GeoJSON ~10-20 MB per miasto, CSV ~2-5 MB per miasto

Wyjście tej sekcji to katalog `outputs/etap3/{city}/od_units/` zawierający gotowe sources i destinations do R5.

In [9]:
OD_META_ROWS = []

od_by_city = {}



for city in CITIES:

    print(f'\n--- {city.upper()} ---')

    city_dir = OUT_DIR / city

    (city_dir / 'od').mkdir(parents=True, exist_ok=True)

    (city_dir / 'ttm').mkdir(parents=True, exist_ok=True)

    (city_dir / 'wtc').mkdir(parents=True, exist_ok=True)

    (city_dir / 'qc').mkdir(parents=True, exist_ok=True)

    (city_dir / 'reports').mkdir(parents=True, exist_ok=True)



    city_key = city.replace('_core', '')



    # --- role modes ---

    if city_key == 'paris':

        if PARIS_OD_ROLE_MODE == 'grid_to_grid':

            origin_role, dest_role = 'grid', 'grid'

        else:

            origin_role, dest_role = 'grid', 'employment'

    else:

        origin_role, dest_role = 'grid', 'grid'



    units_o = load_od_units(city, origin_role)

    units_d = load_od_units(city, dest_role)



    # Filtruj do obszaru obsługi (admin boundary + bliskość przystanków)

    if USE_ADMIN_BOUNDARY_FILTER:

        city_buffer_m = get_stop_buffer_m(city)

        units_o, qc_filter_o = filter_units_to_service_area(units_o, city, buffer_m=city_buffer_m)

        units_d, qc_filter_d = filter_units_to_service_area(units_d, city, buffer_m=city_buffer_m)

        # Zapisz QC filtrowania

        (city_dir / 'qc' / 'od_filter_origins.json').write_text(

            json.dumps(qc_filter_o, ensure_ascii=False, indent=2), encoding='utf-8')

        (city_dir / 'qc' / 'od_filter_destinations.json').write_text(

            json.dumps(qc_filter_d, ensure_ascii=False, indent=2), encoding='utf-8')



    cent_o = build_centroids_with_ids(units_o)

    cent_d = build_centroids_with_ids(units_d)



    # Dla dublin_core ograniczamy puste/peryferyjne komórki do aktywnego rdzenia.

    if city == 'dublin_core':

        def _activity_mask(df: gpd.GeoDataFrame, prefer_col: str) -> pd.Series:

            cols = [prefer_col, 'pop', 'jobs', 'employment', 'tot']

            for col in cols:

                if col in df.columns:

                    s = pd.to_numeric(df[col], errors='coerce').fillna(0)

                    return s > 0

            return pd.Series(True, index=df.index)



        n_o0, n_d0 = len(cent_o), len(cent_d)
        cent_o_pre_activity = cent_o.copy()
        cent_d_pre_activity = cent_d.copy()

        cent_o = cent_o.loc[_activity_mask(cent_o, 'pop')].copy()

        cent_d = cent_d.loc[_activity_mask(cent_d, 'jobs')].copy()

        print(f'  dublin_core: activity filter -> origins {n_o0}->{len(cent_o)}, destinations {n_d0}->{len(cent_d)}')

        # Wycięcie dalszych peryferii: tylko komórki w promieniu od centrum Dublina.
        center_lon, center_lat = DUBLIN_CORE_CENTER_WGS84
        center_pt = gpd.GeoSeries(
            gpd.points_from_xy([center_lon], [center_lat]),
            crs='EPSG:4326'
        ).to_crs(CITY_CRS_METRIC['dublin']).iloc[0]

        cent_o_active = cent_o.copy()
        cent_d_active = cent_d.copy()

        o_dist = cent_o_active.geometry.distance(center_pt)
        d_dist = cent_d_active.geometry.distance(center_pt)
        cent_o = cent_o_active.loc[o_dist <= float(DUBLIN_CORE_MAX_RADIUS_M)].copy()
        cent_d = cent_d_active.loc[d_dist <= float(DUBLIN_CORE_MAX_RADIUS_M)].copy()

        # Gdyby rdzeń był zbyt ciasny dla próbkowania, relaksujemy promień (bez powrotu do obwarzanka).
        if len(cent_o) < OD_SAMPLE_K or len(cent_d) < OD_SAMPLE_K:
            relaxed_r = int(DUBLIN_CORE_MAX_RADIUS_M * 1.1)
            o_dist_r = cent_o_active.geometry.distance(center_pt)
            d_dist_r = cent_d_active.geometry.distance(center_pt)
            cent_o = cent_o_active.loc[o_dist_r <= float(relaxed_r)].copy()
            cent_d = cent_d_active.loc[d_dist_r <= float(relaxed_r)].copy()
            print(f'  dublin_core: radius relaxed {DUBLIN_CORE_MAX_RADIUS_M}m -> {relaxed_r}m to preserve OD sample size')

        # Nie dokładamy jednostek spoza filtra rdzeniowego,
        # żeby dublin_core nie rozszerzał się na peryferie i puste obszary.
        # Jeśli po filtrach jest < OD_SAMPLE_K, sample_units zwróci dostępny podzbiór.

        print(f'  dublin_core: radius filter -> origins {len(cent_o_active)}->{len(cent_o)}, destinations {len(cent_d_active)}->{len(cent_d)}')



    if OD_SCALE_MODE == 'sample_k':

        cent_o_s = sample_units(cent_o, OD_SAMPLE_K, OD_RANDOM_SEED)

        cent_d_s = sample_units(cent_d, OD_SAMPLE_K, OD_RANDOM_SEED)

    elif OD_SCALE_MODE == 'top_cumulative_percentile':

        def keep_top_p(gdf, col, p):

            if col not in gdf.columns: return gdf

            # Drop zero/NaN

            gdf = gdf[gdf[col].notna() & (gdf[col] > 0)].copy()

            # Sort descending

            gdf = gdf.sort_values(by=col, ascending=False)

            # Cumulative sum

            cum = gdf[col].cumsum()

            total = gdf[col].sum()

            # Keep up to threshold

            return gdf[cum <= total * p].copy()



        o_col = CITY_POP_COLUMN.get(city)

        if o_col and o_col in cent_o.columns:

            cent_o_s = keep_top_p(cent_o, o_col, OD_CUMULATIVE_PERCENTILE)

        else:

            cent_o_s = cent_o.copy()



        d_col = CITY_JOBS_COLUMN.get(city)

        if d_col and d_col in cent_d.columns:

            cent_d_s = keep_top_p(cent_d, d_col, OD_CUMULATIVE_PERCENTILE)

        else:

            cent_d_s = cent_d.copy()



        if len(cent_o_s) > OD_SAMPLE_K:

            cent_o_s = sample_units(cent_o_s, OD_SAMPLE_K, OD_RANDOM_SEED)

        if len(cent_d_s) > OD_SAMPLE_K:

            cent_d_s = sample_units(cent_d_s, OD_SAMPLE_K, OD_RANDOM_SEED + 1)

    else:

        raise ValueError(f'Unknown OD_SCALE_MODE: {OD_SCALE_MODE}')



    # Harmonizacja nazw identyfikatorów dla downstream

    origins = cent_o_s.copy()

    destinations = cent_d_s.copy()

    if 'unit_id' in origins.columns and 'origin_id' not in origins.columns:

        origins = origins.rename(columns={'unit_id': 'origin_id'})

    if 'unit_id' in destinations.columns and 'dest_id' not in destinations.columns:

        destinations = destinations.rename(columns={'unit_id': 'dest_id'})



    if 'origin_id' not in origins.columns:

        origins['origin_id'] = [f'o_{i}' for i in range(len(origins))]

    if 'dest_id' not in destinations.columns:

        destinations['dest_id'] = [f'd_{i}' for i in range(len(destinations))]



    # Zapis OD

    origins.to_file(city_dir / 'od' / 'origins.geojson', driver='GeoJSON')

    destinations.to_file(city_dir / 'od' / 'destinations.geojson', driver='GeoJSON')

    pd.DataFrame(origins.drop(columns='geometry')).to_csv(city_dir / 'od' / 'origins_lookup.csv', index=False)

    pd.DataFrame(destinations.drop(columns='geometry')).to_csv(city_dir / 'od' / 'destinations_lookup.csv', index=False)



    od_by_city[city] = {

        'origin_role': origin_role,

        'dest_role': dest_role,

        'origins': origins,

        'destinations': destinations,

    }



    OD_META_ROWS.append({

        'city': city,

        'crs_metric': CITY_CRS_METRIC[city],

        'origin_role': origin_role,

        'dest_role': dest_role,

        'od_scale_mode': OD_SCALE_MODE,

        'n_origins': len(origins),

        'n_destinations': len(destinations),

        'n_pairs': len(origins) * len(destinations),

        'use_admin_boundary_filter': USE_ADMIN_BOUNDARY_FILTER,

        'stops_proximity_buffer_m': get_stop_buffer_m(city) if USE_ADMIN_BOUNDARY_FILTER else None,

        'origins_with_pop': int(pd.to_numeric(origins.get('pop', pd.Series(dtype=float)), errors='coerce').fillna(0).gt(0).sum()) if len(origins) else 0,

        'dests_with_attr': int(pd.to_numeric(destinations.get('jobs', pd.Series(dtype=float)), errors='coerce').fillna(0).gt(0).sum()) if len(destinations) else 0,

    })



od_meta_df = pd.DataFrame(OD_META_ROWS)

od_meta_df.to_csv(OUT_DIR / 'od_summary.csv', index=False)

display(od_meta_df)



--- WARSAW_CORE ---
  warsaw_core: service area filter: 5040 -> 515 (boundary) -> 515 (stops <=3000m)
  warsaw_core: service area filter: 5040 -> 515 (boundary) -> 515 (stops <=3000m)

--- DUBLIN_CORE ---
  dublin_core: service area filter: 7087 -> 984 (boundary) -> 584 (stops <=800m)
  dublin_core: service area filter: 7087 -> 984 (boundary) -> 584 (stops <=800m)
  dublin_core: activity filter -> origins 584->515, destinations 584->580
  dublin_core: radius relaxed 10000m -> 11000m to preserve OD sample size
  dublin_core: radius filter -> origins 515->278, destinations 580->305

--- WARSAW ---
  warsaw: service area filter: 5040 -> 5040 (boundary) -> 2996 (stops <=3000m)
  warsaw: service area filter: 5040 -> 5040 (boundary) -> 2996 (stops <=3000m)

--- PARIS ---
  paris: service area filter: 1444 -> 1444 (boundary) -> 1444 (stops <=3000m)
  paris: service area filter: 1444 -> 1444 (boundary) -> 1444 (stops <=3000m)

--- DUBLIN ---
  dublin: service area filter: 7087 -> 7087 (bounda

,city,crs_metric,origin_role,dest_role,od_scale_mode,n_origins,n_destinations,n_pairs,use_admin_boundary_filter,stops_proximity_buffer_m,origins_with_pop,dests_with_attr
0,warsaw_core,EPSG:2180,grid,grid,sample_k,515,515,265225,True,3000,487,515
1,dublin_core,EPSG:2157,grid,grid,sample_k,278,305,84790,True,800,278,305
2,warsaw,EPSG:2180,grid,grid,sample_k,750,750,562500,True,3000,668,133
3,paris,EPSG:2154,grid,grid,sample_k,750,750,562500,True,3000,750,678
4,dublin,EPSG:2157,grid,grid,sample_k,750,750,562500,True,3000,693,743


## 6) Konfiguracja R5 per miasto (GTFS + OSM)

### Co to R5py?

**Conveyal R5** to open-source engine do routingu transportu zbiorowego. Jest to **engine**, który:
- Wczytuje GTFS (rozkład jazdy, przystanki, trasy)
- Wczytuje mapę uliczną (OpenStreetMap / PBF)
- Oblicza macierze czasu podróży dla par OD, uwzględniając:
  - **First-mile walking** (dojście do przystanku)
  - **Transit routing** (optymalna trasa busami/tramwajami/pociągami)
  - **Waiting time** (oczekiwanie na pojazd - estymowane z headway)
  - **Transfer penalties** (czas przesiadki)
  - **Last-mile walking** (odejście od przystanku do celu)

**R5py** to Python wrapper do R5, obsługuje go poprzez Java (wymaga JRE/JDK). Zabrania czasami zapamiętania macierzy na dysku.

### Konfiguracja per miasto:

**Dublin:**
```python
{
  'city': 'dublin',
  'gtfs_path': 'path/to/GTFS_Realtime2.zip',  # Dublin GTFS
  'osm_path': 'path/to/ireland-and-northern-ireland-latest.osm.pbf',  # całą Irlandia
  'crs': 'EPSG:2157',  # ITM (Irish Transverse Mercator)
  'analysis_zones': ['county_dublin', 'county_kildare', 'county_meath', 'county_wicklow'],
}
```

**Paryż:**
```python
{
  'city': 'paris',
  'gtfs_path': 'path/to/tdg-80921-202604170039.zip',  # MÃ©tropole du Grand Paris GTFS
  'osm_path': 'path/to/france-latest.osm.pbf',  # Francja (lub subset ÃŽle-de-France)
  'crs': 'EPSG:2154',  # Lambert-93
  'analysis_zones': ['Paris', 'VallÃ©e Sud Grand Paris', ...],  # 12 terytoriów EPT
}
```

**Warszawa:**
- **GTFS:** `warsaw_merged.zip` (scalony z ZTM, KM, WKD, Polregio)
  - Jeśli merge się nie powiódł, fallback do poszczególnych ZIP'ów
  - Scalanie dodaje prefiksy do ID (`ztm_stop_id`, `km_stop_id`, itd.) aby uniknąć duplikatów
  - Tworzy "transfers" między operatorami (pieszą odległością â‰¤200 m)

```python
{
  'city': 'warsaw',
  'gtfs_path': 'path/to/warsaw_merged.zip',  # scalony GTFS lub pojedyncze ZIP
  'osm_path': 'path/to/poland-latest.osm.pbf',  # Polska
  'crs': 'EPSG:2180',  # CS92 (PL92)
  'analysis_zones': ['Warsaw', 'Zachodni', 'Pruszkowski', ...],
}
```

### Przygotowanie R5 do obliczeń:

W sekcji **"Budowa i walidacja R5TransitNetwork"**:

1. **Inicjalizacja R5:**
   ```python
   from r5py import TransitNetwork
   network = TransitNetwork.from_file(
     gtfs=Path(...).as_posix(),  # GTFS obowiązkowy
     dem=None,  # Digital Elevation Model (opcjonalnie; tutaj None)
     data_folder=WORK_DIR,  #Cache dla R5 (oszczędza reaglowanie)
   )
   ```

2. **Konfiguracja parametrów podróży:**
   ```python
   travel_time_matrix = network.travel_time_matrix(
       origins=origins_gdf,       # GeoDataFrame z geometrią
       destinations=destinations_gdf,
       departure_time=departure_datetime,  # np. 2016-06-01 07:30:00
       max_walk_distance=MAX_WALK_DISTANCE_M,  # 2000 m
       max_trip_duration=MAX_TRIP_DURATION_MIN,  # 180 min
   )
   ```

3. **Uwzględnianie headway:**
   - R5 automatycznie estymuje waiting time na prze podstawie headway danej linii
   - Jeśli headway = 10 min, średni waiting time â‰ˆ 5 min (headway/2)
   - To jest wbudowane w TTM - nie musimy dodawać ruczniwe

4. **Fallback (jeśli R5 się wysypie):**
   - Jeśli Java się wymięcie, niebiałe inicjacji R5 zwraca None
   - Fallback: obliczamy walk-only (dystans euklidesowy / prędkość spaceru + detour factor)
   - Oznaczamy to w metadanych: `engine_mode = 'walk_only'`

### Uwagi na temat data/czasu:

- R5 wymaga konkretnej daty i godziny (wejście dla kalendarza GTFS i rozkładów)
- Jeśli `ANALYSIS_DATE_YYYYMMDD = None`: dobieramy środek zakresu dostępnych dni w GTFS
- Godzina: jeśli `DEPARTURE_TIME_WINDOW = '06:00:00 - 7:30:00'`, R5 (z `R5_DEPARTURE_TIME_WINDOW_MIN=10`) próbkuje departures w oknie 06:00 Â± 10 min, czyli 5:50â€“6:10, a następnie bierze median/min/max czasów podróży

In [10]:
DATA_DIR = ROOT / 'Data'

# Pozwól nadpisać ścieżki OSM/GTFS przez ENV, żeby łatwo uruchomić r5py bez grzebania w kodzie.

def _env_path(var_name: str) -> Optional[Path]:
    v = os.environ.get(var_name)
    if not v:
        return None
    p = Path(v)
    if not p.is_absolute():
        p = (ROOT / p).resolve()
    return p

CITY_R5_INPUTS = {
    'warsaw_core': {
        'gtfs_zip': _env_path('GTFS_ZIP_WARSAW') or (DATA_DIR / 'Warsaw' / 'warsaw_merged_v4.zip'),
        'osm_pbf': DATA_DIR / 'Warsaw' / 'mazowieckie-260111_v4.osm.pbf'
    },
    'dublin_core': {
        'gtfs_zip': DATA_DIR / 'Dublin' / 'GTFS_Realtime2.zip',
        'osm_pbf': DATA_DIR / 'Dublin' / 'ireland-and-northern-ireland-latest.osm.pbf',
    },
    'dublin': {
        'gtfs_zip': DATA_DIR / 'Dublin' / 'GTFS_Realtime2.zip',
        'osm_pbf': DATA_DIR / 'Dublin' / 'ireland-and-northern-ireland-latest.osm.pbf',
    },
    'paris': {
        'gtfs_zip': DATA_DIR / 'Paris' / 'tdg-80921-202604170039.zip',
        'osm_pbf': DATA_DIR / 'Paris' / 'ile-de-france-260111.osm.pbf',
    },
    'warsaw': {
        # Warszawa: prefer the pre-existing merged feed from Etap I (contains ZTM + KM + WKD + Polregio).
        # The auto-merge from individual ZIPs is a fallback (some ZIPs may be cloud-only/missing).
        'gtfs_zip': _env_path('GTFS_ZIP_WARSAW') or (DATA_DIR / 'Warsaw' / 'warsaw_merged_v4.zip'),
        # 'gtfs_zip_alternatives': [
        #     DATA_DIR / 'Warsaw' / 'warsaw_merged_v4.zip',
        #     DATA_DIR / 'Warsaw' / 'gtfs - ztm wwa.zip',
        #     DATA_DIR / 'Warsaw' / 'kolejemazowieckie.zip',
        #     DATA_DIR / 'Warsaw' / 'wkd.zip',
        #     DATA_DIR / 'Warsaw' / 'GTFS - polregio.zip',
        #     # fallback last-resort (historyczny)
        #     DATA_DIR / 'Warsaw' / 'warsaw -mkuran.zip',
        # ],
        'osm_pbf': DATA_DIR / 'Warsaw' / 'mazowieckie-260111_v4.osm.pbf'
    },
}

# gdzie trzymamy zmergowane feedy GTFS
MERGED_GTFS_DIR = OUT_DIR / '_merged_gtfs'
MERGED_GTFS_DIR.mkdir(parents=True, exist_ok=True)


def _build_merged_warsaw_gtfs(candidate_zips: list[Path]) -> Optional[Path]:
    """Merge multiple Warsaw operator feeds into one GTFS zip."""
    existing = [Path(p) for p in candidate_zips if p and Path(p).exists()]
    if len(existing) < 2:
        return None

    out_zip = MERGED_GTFS_DIR / 'warsaw_merged_gtfs.zip'

    # cache: jeśli out jest nowszy niż inputy, nie miksuj ponownie
    if out_zip.exists():
        out_mtime = out_zip.stat().st_mtime
        if all(p.stat().st_mtime <= out_mtime for p in existing):
            return out_zip

    merge_gtfs_zips(existing, out_zip=out_zip)
    return out_zip


def _pick_best_gtfs_zip(candidate_zips: list[Path]) -> Optional[Path]:
    """Wybiera feed GTFS o możliwie późnej dacie max (aktualność), a potem po długości zakresu."""
    best = None
    best_span = None
    best_max = None

    for zp in candidate_zips:
        if not zp or not Path(zp).exists():
            continue
        try:
            rng = read_gtfs_service_date_range(Path(zp))
        except Exception:
            continue
        if rng.is_empty():
            continue

        span = (rng.max_date() - rng.min_date()).days
        maxd = rng.max_date()

        if best is None:
            best, best_span, best_max = Path(zp), span, maxd
            continue

        if (maxd and best_max and maxd > best_max) or (maxd == best_max and span > best_span):
            best, best_span, best_max = Path(zp), span, maxd

    return best


def _departure_datetimes_for_city(city: str, gtfs_zip: Path, candidates: tuple) -> list[datetime]:
    """Pick one or multiple departure datetimes on an actual GTFS service day."""
    if ANALYSIS_DATE_YYYYMMDD:
        base = ANALYSIS_DATE_YYYYMMDD
    else:
        # Use 'mid' (middle of GTFS range) for robustness — avoids boundary conditions
        # where R5 considers edge dates as "outside time range" due to feed_info.txt constraints.
        base = '20260506' #choose_service_day_within_gtfs(gtfs_zip, prefer_weekdays=True, prefer='mid')

    d = datetime.strptime(base, '%Y%m%d').date()

    if USE_MULTI_DEPARTURES:
        out = []
        for hhmmss in candidates:
            hh, mm, ss = (int(x) for x in hhmmss.split(':'))
            out.append(datetime(d.year, d.month, d.day, hh, mm, ss))
        return out

    hhmmss = candidates[0]
    hh, mm, ss = (int(x) for x in hhmmss.split(':'))
    return [datetime(d.year, d.month, d.day, hh, mm, ss)]


def _aggregate_multi_departure(ttms: list[pd.DataFrame], how: str = 'min') -> pd.DataFrame:
    """Aggregate multiple ttm dataframes into one by OD pair."""
    if not ttms:
        return pd.DataFrame(columns=['origin_id', 'dest_id', 'travel_time_min'])

    base = pd.concat(ttms, ignore_index=True)
    base['travel_time_min'] = pd.to_numeric(base['travel_time_min'], errors='coerce')

    if how == 'median':
        return base.groupby(['origin_id', 'dest_id'], as_index=False).agg(travel_time_min=('travel_time_min', 'median'))

    return base.groupby(['origin_id', 'dest_id'], as_index=False).agg(travel_time_min=('travel_time_min', 'min'))


# Filtr snap R5: weryfikacja, które OD punkty R5 potrafi snap-ować do sieci

def _r5_filter_snappable_points(
    tn,
    pts_wgs84: gpd.GeoDataFrame,
    id_col: str = 'id',
    departure: Optional[datetime] = None,
    require_as_origin_and_dest: bool = True,
) -> tuple[gpd.GeoDataFrame, dict]:
    """Filter points to those that r5 can snap to the street network.

    Works without pyrosm and avoids CRS mismatch issues because we pass WGS84.

    Returns: (pts_kept_wgs84, qc)
    """
    if pts_wgs84 is None or len(pts_wgs84) == 0:
        return pts_wgs84, {'enabled': True, 'reason': 'empty_input', 'n_before': 0, 'n_after': 0}

    if departure is None:
        departure = datetime(2025, 1, 1, 8, 0, 0)

    # small self-matrix (WALK network only)
    try:
        ttm_snap = r5py.TravelTimeMatrix(
            transport_network=tn,
            origins=pts_wgs84[[id_col, 'geometry']].copy(),
            destinations=pts_wgs84[[id_col, 'geometry']].copy(),
            departure=departure,
            departure_time_window=timedelta(minutes=1),
            transport_modes=[r5py.TransportMode.WALK],
            max_time=timedelta(minutes=15),  # zwiększone z 5 min — więcej czasu na snap
            speed_walking=WALK_SPEED_KMH,
            snap_to_network=True,
        )
        if not isinstance(ttm_snap, pd.DataFrame):
            ttm_snap = pd.DataFrame(ttm_snap)
    except Exception as e:
        return pts_wgs84, {'enabled': False, 'reason': f'r5_snap_probe_failed:{e}'}

    if 'from_id' not in ttm_snap.columns or 'to_id' not in ttm_snap.columns:
        return pts_wgs84, {'enabled': False, 'reason': 'r5_snap_probe_missing_columns'}

    from_ids = set(ttm_snap['from_id'].astype('string').unique())
    to_ids = set(ttm_snap['to_id'].astype('string').unique())

    if require_as_origin_and_dest:
        keep = from_ids & to_ids
    else:
        keep = from_ids | to_ids

    before = int(len(pts_wgs84))
    kept = gpd.GeoDataFrame(
        pts_wgs84[pts_wgs84[id_col].astype('string').isin(keep)].copy(),
        geometry='geometry',
        crs=pts_wgs84.crs,
    )
    after = int(len(kept))

    qc = {
        'enabled': True,
        'n_before': before,
        'n_after': after,
        'dropped': before - after,
        'kept_share': float(after / before) if before else 0.0,
        'require_as_origin_and_dest': bool(require_as_origin_and_dest),
    }
    return kept, qc

## 7) Fallback walk-only (jeśli R5 nie działa)

### Scenariusze awaryjne:

1. **Java nie zainstalowana** - R5py nie może się załadować (ImportError)
2. **GTFS lub OSM uszkodzony** - R5 crash'uje przy inicjalizacji
3. **Timeout R5** - obliczenia TTM trwają za długo, system go kill'uje
4. **R5 zwraca empty matrix** - ciche błąd, macierz całkowicie pusta (0 par OD)

### Implementacja fallback'u:

Funkcja `compute_walk_only_ttm(origins, destinations, city, max_time_min=90, detour_factor=1.3)`:

**Algorytm:**
1. Oblicz odległość euklidesową między każdą parą origin-destination
   ```python
   dist_euclidean_m = origin.geometry.distance(dest.geometry)  # w CRS metrycznym
   ```

2. Zastosuj detour factor (1.3 = +30% na zawinięcia drogi)
   ```python
   dist_network_est = dist_euclidean_m * WALK_DETOUR_FACTOR
   ```

3. Konwertuj na czas podróży
   ```python
   travel_time_min = dist_network_est / (WALK_SPEED_MPS * 60)  # km → metrów/s → minuty
   ```

4. **Filtruj:**
   - Jeśli `travel_time_min > FALLBACK_WALK_ONLY_MAX_TIME_MIN` (90 min) → oznacz jako NaN (nieosiągalna)
   - Jeśli `travel_time_min <= 90 min` → zwróć jako normalny czas podróży

   **Uzasadnienie:** Spacer dłuższy niż ~90 minut (>6.5 km) jest praktycznie nierealizowany w warunkach miasta

5. Zwróć DataFrame:
   ```python
   {
     'origin_id': origin ID,
     'dest_id': destination ID,
     'travel_time_min': time (lub NaN),
     'engine': 'walk_only',
     'distance_m': dist_network_est,
   }
   ```

### QC note'y:

- Macierz walk-only jest **rzadka** (sparse) - większość par to NaN, jeśli maksymalny czasy spaceru to 90 min
- Dla porównania R5 (z transportem zbiorowym) zazwyczaj ma więcej niespustych par
- **Zalecenie:** Jeśli > 50% macierzy to walk-only fallback → zaflajuj jako â€žLOW QUALITY" w raporcie
- **Przeznaczenie:** walk-only jest "ostatnią deską ratunku" do analizy; lepiej mieć coś niż nic

### Parametry:

```python
FALLBACK_WALK_ONLY_MAX_TIME_MIN = 90.0  # obniżone z 240 min - >90 min = nieosiągalne
WALK_DETOUR_FACTOR = 1.3  # +30% dystansu euklidesowego
WALK_SPEED_MPS = 1.25  # 4.5 km/h
```

Jeśli TTM z R5 to są pusta lub złe jakości, fallback walk-only zapewnia minimalny poziom danych do dalszej analizy (choć z wyraźnym ostrzeżeniem w metadanych).

In [11]:
def iter_fallback_walk_only_batches(
    origins: gpd.GeoDataFrame,
    destinations: gpd.GeoDataFrame,
    batch_size_origins: int = 60,
):
    """Yield walk-only TTM batches to keep RAM low.

    Each yielded chunk is a DataFrame with columns: origin_id, dest_id, travel_time_min.
    Uses float32 internally to reduce memory pressure.
    Applies WALK_DETOUR_FACTOR to correct euclidean distance to network distance.
    """
    o = origins[['origin_id', 'geometry']].copy()
    d = destinations[['dest_id', 'geometry']].copy()

    d_xy = np.vstack([d.geometry.x.to_numpy(), d.geometry.y.to_numpy()]).T.astype('float32')
    dest_ids = d['dest_id'].to_numpy()
    cap_min = float(FALLBACK_WALK_ONLY_MAX_TIME_MIN)
    detour = float(WALK_DETOUR_FACTOR)

    for start in range(0, len(o), int(batch_size_origins)):
        oo = o.iloc[start:start + int(batch_size_origins)].copy()
        o_xy = np.vstack([oo.geometry.x.to_numpy(), oo.geometry.y.to_numpy()]).T.astype('float32')

        dx = o_xy[:, None, 0] - d_xy[None, :, 0]
        dy = o_xy[:, None, 1] - d_xy[None, :, 1]
        dist_m = np.sqrt(dx * dx + dy * dy, dtype=np.float32) * detour

        t_min = dist_m / float(WALK_SPEED_MPS) / 60.0
        t_min = np.clip(t_min, 0.0, cap_min).astype('float32')

        origin_ids = oo['origin_id'].to_numpy()
        yield pd.DataFrame(
            {
                'origin_id': np.repeat(origin_ids, len(dest_ids)),
                'dest_id': np.tile(dest_ids, len(origin_ids)),
                'travel_time_min': t_min.reshape(-1),
            }
        )


def write_parquet_in_batches(parquet_path: Path, batches) -> None:
    """Write parquet from a generator/iterable of DataFrames without concatenating into RAM."""
    parquet_path.parent.mkdir(parents=True, exist_ok=True)

    try:
        import pyarrow as pa
        import pyarrow.parquet as pq
    except Exception:
        dfs = list(batches)
        if not dfs:
            pd.DataFrame(columns=['origin_id', 'dest_id', 'travel_time_min']).to_parquet(parquet_path, index=False)
            return
        pd.concat(dfs, ignore_index=True).to_parquet(parquet_path, index=False)
        return

    writer = None
    try:
        wrote_any = False
        for b in batches:
            if b is None or len(b) == 0:
                continue
            table = pa.Table.from_pandas(b, preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(parquet_path, table.schema, compression='snappy')
            writer.write_table(table)
            wrote_any = True

        if not wrote_any:
            empty = pa.Table.from_pydict({'origin_id': [], 'dest_id': [], 'travel_time_min': []})
            pq.write_table(empty, parquet_path, compression='snappy')
    finally:
        if writer is not None:
            writer.close()

def _walk_time_minutes_for_pairs(
    origins: gpd.GeoDataFrame,
    destinations: gpd.GeoDataFrame,
    origin_ids: pd.Series,
    dest_ids: pd.Series,
) -> np.ndarray:
    """Compute walk times [min] for given OD id pairs in metric CRS.

    Applies WALK_DETOUR_FACTOR to approximate network distance from euclidean.
    Returns NaN for pairs with unmatched IDs (instead of silently mapping to index 0).
    """
    o = origins[['origin_id', 'geometry']].copy()
    d = destinations[['dest_id', 'geometry']].copy()

    o_xy = np.vstack([o.geometry.x.to_numpy(), o.geometry.y.to_numpy()]).T.astype('float32')
    d_xy = np.vstack([d.geometry.x.to_numpy(), d.geometry.y.to_numpy()]).T.astype('float32')

    o_index = {str(i): idx for idx, i in enumerate(o['origin_id'].astype('string').to_numpy())}
    d_index = {str(i): idx for idx, i in enumerate(d['dest_id'].astype('string').to_numpy())}

    oi = origin_ids.astype('string').map(lambda x: o_index.get(str(x), -1)).to_numpy()
    di = dest_ids.astype('string').map(lambda x: d_index.get(str(x), -1)).to_numpy()

    # Zamiast cichej podmiany -1→0, logujemy i ustawiamy NaN
    bad_mask = (oi < 0) | (di < 0)
    if bad_mask.any():
        n_bad = int(bad_mask.sum())
        print(f'  WARNING: _walk_time_minutes_for_pairs: {n_bad} pairs with unmatched IDs -> NaN')
        oi = np.where(oi < 0, 0, oi)
        di = np.where(di < 0, 0, di)

    detour = float(WALK_DETOUR_FACTOR)
    dx = o_xy[oi, 0] - d_xy[di, 0]
    dy = o_xy[oi, 1] - d_xy[di, 1]
    dist_m = np.sqrt(dx * dx + dy * dy, dtype=np.float32) * detour

    t_min = dist_m / float(WALK_SPEED_MPS) / 60.0
    t_min = np.clip(t_min, 0.0, float(FALLBACK_WALK_ONLY_MAX_TIME_MIN)).astype('float32')

    # Ustaw NaN dla par z nieznalezionymi ID
    if bad_mask.any():
        t_min[bad_mask] = np.nan

    return t_min


def _fill_missing_pairs_with_walk(
    ttm: pd.DataFrame,
    origins: gpd.GeoDataFrame,
    destinations: gpd.GeoDataFrame,
) -> pd.DataFrame:
    """Complete the OD matrix by filling missing travel_time_min with walk-only times.

    - Keeps any available R5 `travel_time_min` values.
    - Fills only missing pairs with network-corrected walk time.
    - Assumes `origins`/`destinations` are metric CRS.

    Returns a full |O|x|D| DataFrame with columns: origin_id, dest_id, travel_time_min.
    """
    o_ids = origins['origin_id'].astype('string').to_numpy()
    d_ids = destinations['dest_id'].astype('string').to_numpy()
    full = pd.MultiIndex.from_product([o_ids, d_ids], names=['origin_id', 'dest_id']).to_frame(index=False)

    if ttm is None or len(ttm) == 0:
        merged = full
        merged['travel_time_min'] = np.nan
    else:
        base = ttm[['origin_id', 'dest_id', 'travel_time_min']].copy()
        base['origin_id'] = base['origin_id'].astype('string')
        base['dest_id'] = base['dest_id'].astype('string')
        merged = full.merge(base, on=['origin_id', 'dest_id'], how='left')

    miss = merged['travel_time_min'].isna()
    merged['is_walk_fill'] = miss
    if not bool(miss.any()):
        return merged

    merged.loc[miss, 'travel_time_min'] = _walk_time_minutes_for_pairs(
        origins=origins,
        destinations=destinations,
        origin_ids=merged.loc[miss, 'origin_id'],
        dest_ids=merged.loc[miss, 'dest_id'],
    )
    return merged

## 8) Obliczanie macierzy TTM per miasto

### Logika główna (per każde miasto):

**Krok 1: Przygotowanie wejść**
- Załaduj OD units (origins i destinations) z kroku 5
- Załaduj artefakty z Etapu I (GTFS, OSM, granice administracyjne)
- Określ dateę analizy: `ANALYSIS_DATE_YYYYMMDD` lub auto-select ze średka zakresu GTFS

**Krok 2: Próba R5 (jeśli dostępny)**
```python
if HAS_R5PY:
    try:
        network = r5py.TransitNetwork.from_file(
            gtfs=cin.gtfs_tables['stops'],  # PBF OSM
            ...
        )
        ttm_r5 = network.travel_time_matrix(
            origins=origins_gdf,
            destinations=destinations_gdf,
            departure_time=datetime(...),
            max_walk_distance=MAX_WALK_DISTANCE_M,
            ...
        )
        engine_mode = 'r5'
    except Exception as e:
        print(f"  R5 FAILED: {e}")
        ttm_r5 = None
        engine_mode = None
else:
    ttm_r5 = None
```

**Krok 3: QC na TTM z R5 (jeśli udało się)**
```python
if ttm_r5 is not None:
    is_suspicious = _is_suspicious_constant_times(ttm_r5, debug=True)
    nonnull_count = ttm_r5['travel_time_min'].notna().sum()
    nonnull_share = nonnull_count / len(ttm_r5)

    if is_suspicious or nonnull_share < R5_MIN_NONNULL_SHARE:
        print(f"  WARNING: TTM failed QC checks (constant={is_suspicious}, nonnull={nonnull_share})")
        ttm_r5 = None
        engine_mode = None
```

**Krok 4: Fallback walk-only (jeśli R5 nie działał lub nie przeszedł QC)**
```python
if ttm_r5 is None:
    ttm_walk = compute_walk_only_ttm(origins, destinations, city,
                                     max_time_min=FALLBACK_WALK_ONLY_MAX_TIME_MIN,
                                     detour_factor=WALK_DETOUR_FACTOR)
    if FILL_MISSING_WITH_WALK and ttm_r5 is not None:
        # Jeśli R5 częściowo działał, uzupełniamy brakujące pary (NaN) z walk-only
        mask_null = ttm_r5['travel_time_min'].isna()
        ttm_r5.loc[mask_null, 'travel_time_min'] = ttm_walk.loc[mask_null, 'travel_time_min']
        engine_mode = 'r5_with_walk_fill'
    else:
        ttm_r5 = ttm_walk
        engine_mode = 'walk_only'
```

**Krok 5: Normalizacja i czyszczenie TTM**
- Upewnij się, że kolumny są numeryczne
- Zaokrągl czasy do 1 minuty (nie ma sensu dokładność poniżej 1 min w transporcie zbiorowym)
- Dodaj metadane: `engine_mode`, `computed_utc`, `city`, `analysis_date`

**Krok 6: Zapis**
```python
# TTM jako Parquet (kompresja, szybkie read/write)
output_path = OUT_DIR / city / 'ttm_travel_times.parquet'
ttm.to_parquet(output_path)

# Oszczędzaj losowy sample do CSV dla vizualizacji
ttm.sample(min(1000, len(ttm))).to_csv(OUT_DIR / city / 'ttm_sample_1k.csv')
```

### Czasochłonność:

- **Per miasto, R5:** ~10-30 minut (w zależ. od rozmiaru 750x750 OD)
- **Per miasto, walk-only:** ~1 sekunda
- **Razem 3 miasta:** ~1-2 godziny (R5) lub ~3 sekundy (walk-only)

### Metadane:

```python
{
  'city': 'warsaw',
  'engine_mode': 'r5'  # lub 'walk_only' czy 'r5_with_walk_fill',
  'analysis_date': '2016-06-01',
  'n_origins': 750,
  'n_destinations': 750,
  'n_total_pairs': 562500,
  'n_nonnull_pairs': 560000,  # np. 99.6%
  'nonnull_share': 0.996,
  'max_travel_time_min': 180,
  'min_travel_time_min': 2,
  'mean_travel_time_min': 45.3,
  'computed_utc': '2026-04-24T14:30:00Z',
  'version': 'etap3_v2.1',
}
```

**UWAGA:** Ten krok wymaga **Java** (dla R5) i może być **czasochłonny** (szczególnie dla dużych OD). Rekomendujemy uruchamianie w tle lub na high-end maszynie.

In [12]:
# First, let's check GTFS date ranges for all cities
print('=== GTFS Date Ranges & Departure Selection ===')
for city in CITIES:
    r5_inputs = CITY_R5_INPUTS[city]
    gtfs_zip = r5_inputs.get('gtfs_zip')

    if 'warsaw' in city and (not gtfs_zip or not Path(gtfs_zip).exists()):
        alts = r5_inputs.get('gtfs_zip_alternatives', [])
        existing_alts = [Path(p) for p in alts if p and Path(p).exists()]
        if len(existing_alts) >= 2:
            merged = _build_merged_warsaw_gtfs(existing_alts)
            if merged:
                gtfs_zip = merged
        if not gtfs_zip or not Path(gtfs_zip).exists():
            gtfs_zip = _pick_best_gtfs_zip(existing_alts) if existing_alts else None

    if gtfs_zip and Path(gtfs_zip).exists():
        try:
            rng = read_gtfs_service_date_range(Path(gtfs_zip))
            # Also compute what departure day will be chosen
            # chosen_day = choose_service_day_within_gtfs(gtfs_zip, prefer_weekdays=True, prefer='mid')
            chosen_day = '20260506'
            chosen_date = datetime.strptime(chosen_day, '%Y%m%d').date()
            in_range = (rng.min_date() <= chosen_date <= rng.max_date()) if not rng.is_empty() else False
            status = '✅' if in_range else '❌ OUT-OF-RANGE'
            print(f'{city.upper():10s}: GTFS range {rng.min_yyyymmdd} to {rng.max_yyyymmdd} | chosen departure day: {chosen_day} {status}')
            print(f'{"":10s}  has_calendar={rng.has_calendar}, has_calendar_dates={rng.has_calendar_dates}')
            # Check service on chosen day
            has_svc = gtfs_has_any_service_on_date(gtfs_zip, chosen_day)
            print(f'{"":10s}  service active on {chosen_day}: {has_svc}')
            if not has_svc:
                print(f'  ⚠️ WARNING: No GTFS service on chosen day {chosen_day}! R5 will route walk-only.')
        except Exception as e:
            print(f'{city.upper():10s}: Error reading dates - {e}')
    else:
        print(f'{city.upper():10s}: GTFS not found')

print()


=== GTFS Date Ranges & Departure Selection ===
WARSAW_CORE: GTFS range 20260422 to 20260522 | chosen departure day: 20260506 âœ…
            has_calendar=False, has_calendar_dates=True
            service active on 20260506: True
DUBLIN_CORE: GTFS range 20260415 to 20270416 | chosen departure day: 20260506 âœ…
            has_calendar=True, has_calendar_dates=True
            service active on 20260506: True
WARSAW    : GTFS range 20260422 to 20260522 | chosen departure day: 20260506 âœ…
            has_calendar=False, has_calendar_dates=True
            service active on 20260506: True
PARIS     : GTFS range 20260413 to 20260515 | chosen departure day: 20260506 âœ…
            has_calendar=True, has_calendar_dates=True
            service active on 20260506: True
DUBLIN    : GTFS range 20260415 to 20270416 | chosen departure day: 20260506 âœ…
            has_calendar=True, has_calendar_dates=True
            service active on 20260506: True



In [13]:
import sys
if '-Xmx10G' not in sys.argv:
    sys.argv.append('-Xmx10G')

TTM_QC_ROWS = []
WTC_QC_ROWS = []
# For backward compatibility if needed:
ttm_by_city_profile = {}

for city in CITIES:
    print(f'\n=== {city.upper()} ===')
    city_dir = OUT_DIR / city
    od_data = od_by_city[city]
    origins = od_data['origins']
    destinations = od_data['destinations']

    # NEW: diagnostic mode: destinations come from origins (O->O)
    if DIAG_DESTINATIONS_FROM_ORIGINS:
        diag_o = origins.copy()
        if len(diag_o) > int(DIAG_SAMPLE_K):
            diag_o = diag_o.sample(n=int(DIAG_SAMPLE_K), random_state=OD_RANDOM_SEED).copy()
        destinations = diag_o.rename(columns={'origin_id': 'dest_id'}).copy()
        destinations = gpd.GeoDataFrame(destinations, geometry='geometry', crs=origins.crs)
        print(f'DIAG_DESTINATIONS_FROM_ORIGINS=True -> using destinations from origins sample: |O|={len(diag_o)} |D|={len(destinations)}')

    # Keep original OD sets for filling/statistics
    origins_full = origins.copy()
    destinations_full = destinations.copy()


    # Try R5 if available
    if HAS_R5PY:
        r5_inputs = CITY_R5_INPUTS[city]
        gtfs_zip = r5_inputs.get('gtfs_zip')
        osm_pbf = r5_inputs.get('osm_pbf')

        # Warsaw special case: if primary gtfs_zip doesn't exist, merge or pick best from alternatives
        if 'warsaw' in city and (not gtfs_zip or not Path(gtfs_zip).exists()):
            alts = r5_inputs.get('gtfs_zip_alternatives', [])
            existing_alts = [Path(p) for p in alts if p and Path(p).exists()]
            print(f'Warsaw primary GTFS not found ({gtfs_zip}), checking {len(existing_alts)} alternatives...')
            if len(existing_alts) >= 2:
                merged = _build_merged_warsaw_gtfs(existing_alts)
                if merged:
                    gtfs_zip = merged
                    print(f'Using auto-merged GTFS: {gtfs_zip}')
            if not gtfs_zip or not Path(gtfs_zip).exists():
                gtfs_zip = _pick_best_gtfs_zip(existing_alts)
                if gtfs_zip:
                    print(f'Using best single GTFS: {gtfs_zip}')
        elif 'warsaw' in city and gtfs_zip and Path(gtfs_zip).exists():
            print(f'Using primary GTFS: {gtfs_zip}')

        if gtfs_zip and gtfs_zip.exists() and osm_pbf and osm_pbf.exists():
            print(f'\nBuilding R5 transport network for {city} (reused across profiles)...')
            try:
                tn = r5py.TransportNetwork(osm_pbf=osm_pbf, gtfs=[gtfs_zip])
            except Exception as e:
                print(f'Failed to build TransportNetwork for {city}: {e}')
                tn = None
    for profile_name, candidates in TEMPORAL_PROFILES.items():
        print(f'\n  --- PROFILE: {profile_name} ---')

        # Create profile specific subdirectories
        profile_ttm_dir = city_dir / 'ttm' / profile_name
        profile_ttm_dir.mkdir(parents=True, exist_ok=True)
        # We will also need profile WTC and QC potentially, but let's just namespace the files.

        engine_mode = 'walk_only_fallback'  # default
        departures_used: list = []  # track for reproducibility
        warnings = []
        ttm = None

        # Try R5 if available
        if tn is not None:
            try:
                # Service-day-aware departure selection (optionally multi-departure)
                departures = _departure_datetimes_for_city(city, gtfs_zip, candidates)
                departures_used = [str(d) for d in departures]
                print(f'Departures: {departures} (service-day-aware)')

                if True:  # just to keep indentation

                    # Prepare origins/destinations for r5py (must be in WGS84)
                    o_wgs84_full = gpd.GeoDataFrame(
                        origins.to_crs('EPSG:4326')[['origin_id', 'geometry']].rename(columns={'origin_id': 'id'}),
                        geometry='geometry',
                        crs='EPSG:4326',
                    )
                    d_wgs84_full = gpd.GeoDataFrame(
                        destinations.to_crs('EPSG:4326')[['dest_id', 'geometry']].rename(columns={'dest_id': 'id'}),
                        geometry='geometry',
                        crs='EPSG:4326',
                    )

                    # Hard filter to truly snappable street points using r5 itself
                    if SNAP_TO_STREETS_BEFORE_R5:
                        probe_departure = departures[0] if departures else datetime(2025, 1, 1, 8, 0, 0)

                        o_wgs84, qc_o = _r5_filter_snappable_points(tn, o_wgs84_full, id_col='id', departure=probe_departure)
                        d_wgs84, qc_d = _r5_filter_snappable_points(tn, d_wgs84_full, id_col='id', departure=probe_departure)

                        o_wgs84 = gpd.GeoDataFrame(o_wgs84, geometry='geometry', crs='EPSG:4326')
                        d_wgs84 = gpd.GeoDataFrame(d_wgs84, geometry='geometry', crs='EPSG:4326')

                        (city_dir / 'qc' / f'r5_snap_filter_origins_{profile_name}.json').write_text(json.dumps(qc_o, indent=2), encoding='utf-8')
                        (city_dir / 'qc' / f'r5_snap_filter_destinations_{profile_name}.json').write_text(json.dumps(qc_d, indent=2), encoding='utf-8')

                        if qc_o.get('enabled') and qc_o.get('kept_share', 1.0) < float(SNAP_MIN_KEEP_SHARE):
                            warnings.append(f"R5 snap-filter would drop too many origins (kept_share={qc_o.get('kept_share'):.2f}); proceeding without filtering")
                            o_wgs84 = o_wgs84_full
                        if qc_d.get('enabled') and qc_d.get('kept_share', 1.0) < float(SNAP_MIN_KEEP_SHARE):
                            warnings.append(f"R5 snap-filter would drop too many destinations (kept_share={qc_d.get('kept_share'):.2f}); proceeding without filtering")
                            d_wgs84 = d_wgs84_full

                        try:
                            gpd.GeoDataFrame(o_wgs84, geometry='geometry', crs='EPSG:4326').to_file(
                                city_dir / 'od' / f'origins_r5_snapped_{profile_name}.geojson', driver='GeoJSON'
                            )
                            gpd.GeoDataFrame(d_wgs84, geometry='geometry', crs='EPSG:4326').to_file(
                                city_dir / 'od' / f'destinations_r5_snapped_{profile_name}.geojson', driver='GeoJSON'
                            )
                        except Exception:
                            pass
                    else:
                        o_wgs84, d_wgs84 = o_wgs84_full, d_wgs84_full

                    # Compute one or multiple TTMs and aggregate
                    _max_walk_time_min = int(MAX_WALK_DISTANCE_M / WALK_SPEED_MPS / 60) + 5

                    ttms = []
                    for departure in departures:
                        print(f'Computing travel times for {len(o_wgs84)} origins x {len(d_wgs84)} destinations at {departure}...')
                        ttm_raw = r5py.TravelTimeMatrix(
                            transport_network=tn,
                            origins=o_wgs84,
                            destinations=d_wgs84,
                            departure=departure,
                            departure_time_window=timedelta(minutes=int(R5_DEPARTURE_TIME_WINDOW_MIN)),
                            transport_modes=[r5py.TransportMode.TRANSIT],
                            access_modes=[r5py.TransportMode.WALK],
                            egress_modes=[r5py.TransportMode.WALK],
                            max_time=timedelta(minutes=int(MAX_TRIP_DURATION_MIN)),
                              percentiles=[50],  # explicitly request median to avoid multi-array bugs
                            max_time_walking=timedelta(minutes=_max_walk_time_min),
                            speed_walking=WALK_SPEED_KMH,
                            snap_to_network=True,
                        )

                        if not isinstance(ttm_raw, pd.DataFrame):
                            ttm_raw = pd.DataFrame(ttm_raw)

                        ttmp = ttm_raw.rename(columns={'from_id': 'origin_id', 'to_id': 'dest_id', 'travel_time': 'travel_time_min'})
                        ttmp['travel_time_min'] = pd.to_numeric(ttmp['travel_time_min'], errors='coerce')

                        ttms.append(ttmp[['origin_id','dest_id','travel_time_min']])

                    if len(ttms) > 1:
                        ttm = _aggregate_multi_departure(ttms, how=MULTI_DEPARTURE_AGG)
                        print(f'Aggregated multi-departure TTM rows: {len(ttm)}')
                    else:
                        ttm = ttms[0]

                    print(f'After conversion: {len(ttm)} rows, non-null travel_time: {pd.to_numeric(ttm["travel_time_min"], errors="coerce").notna().sum()}')

                    # QC checks
                    is_suspicious = _is_suspicious_constant_times(ttm, debug=True)
                    nonnull_cnt = int(pd.to_numeric(ttm['travel_time_min'], errors='coerce').notna().sum())
                    nonnull_share = float(pd.to_numeric(ttm['travel_time_min'], errors='coerce').notna().mean())
                    is_sparse = (nonnull_share < float(R5_MIN_NONNULL_SHARE)) and (nonnull_cnt < int(R5_MIN_NONNULL_ABS))

                    print(f'  QC sparse check: nonnull_share={nonnull_share:.3f}, threshold={R5_MIN_NONNULL_SHARE}; nonnull_cnt={nonnull_cnt}, min_abs={R5_MIN_NONNULL_ABS}')
                    print(f'QC checks: suspicious_constant={is_suspicious}, too_sparse={is_sparse}')

                    if is_suspicious:
                        warnings.append('R5 returned suspicious constant times')
                        print('  -> Rejecting: suspicious constant times')
                        ttm = None
                    elif is_sparse:
                        warnings.append('R5 TTM extremely sparse (will fallback)')
                        print('  -> Rejecting: extremely sparse')
                        ttm = None
                    else:
                        engine_mode = 'r5py'
                        print(f'R5 TTM accepted: {len(ttm)} rows (nonnull={nonnull_cnt}, share={nonnull_share:.4f})')

                        # POST-HOC TRANSIT VALIDATION
                        s_check = pd.to_numeric(ttm['travel_time_min'], errors='coerce').dropna()
                        median_tt = float(s_check.median()) if len(s_check) else 0.0
                        print(f'  Transit sanity: median_travel_time={median_tt:.1f} min, nonnull={len(s_check)}')

                        try:
                            _rng_check = read_gtfs_service_date_range(gtfs_zip)
                            if not _rng_check.is_empty():
                                _dep_date = departures[0].date() if departures else None
                                _min_d = _rng_check.min_date()
                                _max_d = _rng_check.max_date()
                                if _dep_date and (_dep_date < _min_d or _dep_date > _max_d):
                                    warnings.append(
                                        f'CRITICAL: Departure date {_dep_date} is outside GTFS range '
                                        f'[{_min_d}, {_max_d}]. Results are likely WALK-ONLY, not transit!'
                                    )
                                    print(f'  CRITICAL: departure {_dep_date} outside GTFS [{_min_d},{_max_d}]!')
                        except Exception:
                            pass

                        if DROP_UNSNAPPED_POINTS:
                            kept_o = set(ttm['origin_id'].astype('string').unique())
                            kept_d = set(ttm['dest_id'].astype('string').unique())
                            n_o0, n_d0 = len(origins), len(destinations)
                            origins = origins[origins['origin_id'].astype('string').isin(kept_o)].copy()
                            destinations = destinations[destinations['dest_id'].astype('string').isin(kept_d)].copy()
                            if len(origins) != n_o0 or len(destinations) != n_d0:
                                warnings.append(f'Dropped unsnapped OD points: origins {n_o0}->{len(origins)}, dest {n_d0}->{len(destinations)}')
                                print(f'Dropped unsnapped points: origins {n_o0}->{len(origins)}, dest {n_d0}->{len(destinations)}')

                        # Complete matrix with walk fallback (keeps R5 values where available)
                        if FILL_MISSING_WITH_WALK:
                            fill_origins = origins_full
                            fill_destinations = destinations_full

                            expected_pairs = int(len(fill_origins) * len(fill_destinations))
                            computed_pairs = int(len(ttm))

                            print(f'Fill plan: |O|={len(fill_origins)}, |D|={len(fill_destinations)}, expected_pairs={expected_pairs}, r5_rows={computed_pairs}, r5_nonnull={nonnull_cnt}')

                            ttm_filled = _fill_missing_pairs_with_walk(ttm, fill_origins, fill_destinations)

                            sfill = pd.to_numeric(ttm_filled['travel_time_min'], errors='coerce')
                            filled_nonnull = int(sfill.notna().sum())
                            filled_by_walk = int(filled_nonnull - nonnull_cnt)
                            fill_fraction = float(filled_by_walk / filled_nonnull) if filled_nonnull else 1.0

                            print(f'Fill stats: filled_nonnull={filled_nonnull}, filled_by_walk~={filled_by_walk}, fill_fraction~={fill_fraction:.3f}')

                            if fill_fraction > float(MAX_FILL_FRACTION):
                                warnings.append(f'Fill fraction {fill_fraction:.3f} > {MAX_FILL_FRACTION}; keeping sparse R5 matrix (no fill) to avoid walk-only dominance')
                                print(f'Fill fraction too high -> NOT filling to avoid walk-only dominance')
                                engine_mode = 'r5py_sparse'
                            else:
                                ttm = ttm_filled
                                engine_mode = 'r5py_plus_walk_fill'
                                print(f'Filled matrix size: {len(ttm)} rows')

            except Exception as e:
                warnings.append(f'R5 failed: {e}')
                print(f'R5 failed: {e}')
                ttm = None

        # Fallback to walk-only if R5 not available or failed
        if ttm is None:
            print(f'Using walk-only fallback for {city} [{profile_name}]')
            warnings.append(f'Using walk-only fallback for {profile_name} (R5 not available or failed)')
            engine_mode = f'walk_only_fallback'

            batches = iter_fallback_walk_only_batches(origins, destinations)
            ttm_path = profile_ttm_dir / f'ttm_raw_{profile_name}.parquet'
            write_parquet_in_batches(ttm_path, batches)
            ttm = pd.read_parquet(ttm_path)

        # Write TTM (always overwrite to ensure latest results)
        if ttm is not None:
            ttm.to_parquet(profile_ttm_dir / f'ttm_raw_{profile_name}.parquet', index=False)
            ttm_by_city_profile[f"{city}_{profile_name}"] = ttm

        # QC summary
        # --- DESCRIPTIVE STATS: Calculate only using OD pairs within 3 km of stops ---
        if ttm is not None and STOPS_PROXIMITY_BUFFER_M > 0:
            print(f'\n>> QC Stats: Filtering to O-D pairs within {STOPS_PROXIMITY_BUFFER_M}m of gtfs stops...')
            # filter requires 'unit_id' column
            o_temp = origins.rename(columns={'origin_id':'unit_id'})
            d_temp = destinations.rename(columns={'dest_id':'unit_id'})
            _ou, _ = filter_units_to_service_area(o_temp, city, buffer_m=STOPS_PROXIMITY_BUFFER_M)
            _du, _ = filter_units_to_service_area(d_temp, city, buffer_m=STOPS_PROXIMITY_BUFFER_M)
            valid_o = _ou['unit_id'].unique()
            valid_d = _du['unit_id'].unique()

            mask = ttm['origin_id'].isin(valid_o) & ttm['dest_id'].isin(valid_d)
            ttm_qc = ttm.loc[mask].copy()
            print(f'>> QC Stats: {len(ttm_qc)} valid pairs out of {len(ttm)} total for {city}\n')
        else:
            ttm_qc = ttm if ttm is not None else None

        s = pd.to_numeric(ttm_qc['travel_time_min'], errors='coerce') if ttm_qc is not None else pd.Series(dtype=float)
        qc_row = {
            'city': city,
            'profile': profile_name,
            'engine_mode': engine_mode,
            'matrix_rows': len(ttm) if ttm is not None else 0,
            'non_null_rows': int(s.notna().sum()) if ttm is not None else 0,
            'fill_fraction': fill_fraction if 'fill_fraction' in locals() and ttm is not None and 'r5' in engine_mode else None,
            'min_time': float(s.min()) if (ttm is not None and not s.dropna().empty) else None,
            'median_time': float(s.median()) if (ttm is not None and not s.dropna().empty) else None,
            'max_time': float(s.max()) if (ttm is not None and not s.dropna().empty) else None,
            'departures_used': str(departures_used),
            'warnings': '; '.join(warnings) if warnings else None
        }
        TTM_QC_ROWS.append(qc_row)

        print(f'Finished {city} [{profile_name}]: engine_mode={engine_mode}')

        # Save warnings if any
        if warnings:
            warnings_path = city_dir / 'qc' / f"warnings_{profile_name}.md"
            with open(warnings_path, 'w', encoding='utf-8') as wf:
                wf.write(f"# Warnings for {city} [{profile_name}]\n\n")
                for w in warnings:
                    wf.write(f"- {w}\n")


print('\nDone computing travel time matrices for all cities and profiles.')

# Save TTM_QC summary per city and also global
df_qc = pd.DataFrame(TTM_QC_ROWS)
for city in CITIES:
    df_city = df_qc[df_qc['city'] == city]
    # save overall to ttm_qc_summary.csv so old cells don't break
    df_city.to_csv(OUT_DIR / city / 'ttm' / 'ttm_qc_summary.csv', index=False)


    # Cleanup R5 network to free Java heap memory
    if 'tn' in globals() or 'tn' in locals():
        tn = None
    import gc
    gc.collect()





=== WARSAW_CORE ===
Using primary GTFS: C:\Users\Michc\Dropbox\IO_UW\Magisterka\Masters\Data\Warsaw\warsaw_merged_v4.zip

Building R5 transport network for warsaw_core (reused across profiles)...

  --- PROFILE: morning_peak ---
Departures: [datetime.datetime(2026, 5, 6, 6, 0)] (service-day-aware)
Computing travel times for 515 origins x 515 destinations at 2026-05-06 06:00:00...


C:\Users\Michc\Dropbox\IO_UW\Magisterka\Masters\.venv\lib\site-packages\r5py\r5\regional_task.py:227: RuntimeWarning: Departure time 2026-05-06 06:00:00 is outside of the time range covered by currently loaded GTFS data sets.
  warnings.warn(


After conversion: 265225 rows, non-null travel_time: 262508

>> QC Stats: Filtering to O-D pairs within 3000m of gtfs stops...
  warsaw_core: service area filter: 515 -> 515 (boundary) -> 515 (stops <=3000m)
  warsaw_core: service area filter: 515 -> 515 (boundary) -> 515 (stops <=3000m)
>> QC Stats: 265225 valid pairs out of 265225 total for warsaw_core

  QC constant check: top1_share=0.018, threshold=0.98, top_value=66.0
  QC sparse check: nonnull_share=0.990, threshold=0.01; nonnull_cnt=262508, min_abs=500
QC checks: suspicious_constant=False, too_sparse=False
R5 TTM accepted: 265225 rows (nonnull=262508, share=0.9898)
  Transit sanity: median_travel_time=67.0 min, nonnull=262508
Fill plan: |O|=515, |D|=515, expected_pairs=265225, r5_rows=265225, r5_nonnull=262508
Fill stats: filled_nonnull=265225, filled_by_walk~=2717, fill_fraction~=0.010
Filled matrix size: 265225 rows

>> QC Stats: Filtering to O-D pairs within 3000m of gtfs stops...
  warsaw_core: service area filter: 515 -> 5

C:\Users\Michc\Dropbox\IO_UW\Magisterka\Masters\.venv\lib\site-packages\r5py\r5\regional_task.py:227: RuntimeWarning: Departure time 2026-05-06 11:00:00 is outside of the time range covered by currently loaded GTFS data sets.
  warnings.warn(


After conversion: 265225 rows, non-null travel_time: 261640

>> QC Stats: Filtering to O-D pairs within 3000m of gtfs stops...
  warsaw_core: service area filter: 515 -> 515 (boundary) -> 515 (stops <=3000m)
  warsaw_core: service area filter: 515 -> 515 (boundary) -> 515 (stops <=3000m)
>> QC Stats: 265225 valid pairs out of 265225 total for warsaw_core

  QC constant check: top1_share=0.018, threshold=0.98, top_value=71.0
  QC sparse check: nonnull_share=0.986, threshold=0.01; nonnull_cnt=261640, min_abs=500
QC checks: suspicious_constant=False, too_sparse=False
R5 TTM accepted: 265225 rows (nonnull=261640, share=0.9865)
  Transit sanity: median_travel_time=70.0 min, nonnull=261640
Fill plan: |O|=515, |D|=515, expected_pairs=265225, r5_rows=265225, r5_nonnull=261640
Fill stats: filled_nonnull=265225, filled_by_walk~=3585, fill_fraction~=0.014
Filled matrix size: 265225 rows

>> QC Stats: Filtering to O-D pairs within 3000m of gtfs stops...
  warsaw_core: service area filter: 515 -> 5

...

After conversion: 562500 rows, non-null travel_time: 556522

>> QC Stats: Filtering to O-D pairs within 3000m of gtfs stops...
  paris: service area filter: 750 -> 750 (boundary) -> 750 (stops <=3000m)
  paris: service area filter: 750 -> 750 (boundary) -> 750 (stops <=3000m)
>> QC Stats: 562500 valid pairs out of 562500 total for paris

  QC constant check: top1_share=0.020, threshold=0.98, top_value=75.0
  QC sparse check: nonnull_share=0.989, threshold=0.01; nonnull_cnt=556522, min_abs=500
QC checks: suspicious_constant=False, too_sparse=False
R5 TTM accepted: 562500 rows (nonnull=556522, share=0.9894)
  Transit sanity: median_travel_time=72.0 min, nonnull=556522
Fill plan: |O|=750, |D|=750, expected_pairs=562500, r5_rows=562500, r5_nonnull=556522
Fill stats: filled_nonnull=562500, filled_by_walk~=5978, fill_fraction~=0.011
Filled matrix size: 562500 rows

>> QC Stats: Filtering to O-D pairs within 3000m of gtfs stops...
  paris: service area filter: 750 -> 750 (boundary) -> 750 (st

C:\Users\Michc\Dropbox\IO_UW\Magisterka\Masters\.venv\lib\site-packages\r5py\r5\regional_task.py:227: RuntimeWarning: Departure time 2026-05-06 11:00:00 is outside of the time range covered by currently loaded GTFS data sets.
  warnings.warn(


After conversion: 562500 rows, non-null travel_time: 555879

>> QC Stats: Filtering to O-D pairs within 3000m of gtfs stops...
  paris: service area filter: 750 -> 750 (boundary) -> 750 (stops <=3000m)
  paris: service area filter: 750 -> 750 (boundary) -> 750 (stops <=3000m)
>> QC Stats: 562500 valid pairs out of 562500 total for paris

  QC constant check: top1_share=0.019, threshold=0.98, top_value=73.0
  QC sparse check: nonnull_share=0.988, threshold=0.01; nonnull_cnt=555879, min_abs=500
QC checks: suspicious_constant=False, too_sparse=False
R5 TTM accepted: 562500 rows (nonnull=555879, share=0.9882)
  Transit sanity: median_travel_time=73.0 min, nonnull=555879
Fill plan: |O|=750, |D|=750, expected_pairs=562500, r5_rows=562500, r5_nonnull=555879
Fill stats: filled_nonnull=562500, filled_by_walk~=6621, fill_fraction~=0.012
Filled matrix size: 562500 rows

>> QC Stats: Filtering to O-D pairs within 3000m of gtfs stops...
  paris: service area filter: 750 -> 750 (boundary) -> 750 (st

C:\Users\Michc\Dropbox\IO_UW\Magisterka\Masters\.venv\lib\site-packages\r5py\r5\regional_task.py:227: RuntimeWarning: Departure time 2026-05-06 19:00:00 is outside of the time range covered by currently loaded GTFS data sets.
  warnings.warn(


After conversion: 562500 rows, non-null travel_time: 556001

>> QC Stats: Filtering to O-D pairs within 3000m of gtfs stops...
  paris: service area filter: 750 -> 750 (boundary) -> 750 (stops <=3000m)
  paris: service area filter: 750 -> 750 (boundary) -> 750 (stops <=3000m)
>> QC Stats: 562500 valid pairs out of 562500 total for paris

  QC constant check: top1_share=0.019, threshold=0.98, top_value=75.0
  QC sparse check: nonnull_share=0.988, threshold=0.01; nonnull_cnt=556001, min_abs=500
QC checks: suspicious_constant=False, too_sparse=False
R5 TTM accepted: 562500 rows (nonnull=556001, share=0.9884)
  Transit sanity: median_travel_time=71.0 min, nonnull=556001
Fill plan: |O|=750, |D|=750, expected_pairs=562500, r5_rows=562500, r5_nonnull=556001
Fill stats: filled_nonnull=562500, filled_by_walk~=6499, fill_fraction~=0.012
Filled matrix size: 562500 rows

>> QC Stats: Filtering to O-D pairs within 3000m of gtfs stops...
  paris: service area filter: 750 -> 750 (boundary) -> 750 (st

## 9) Transformacja TTM → WTC (Whole Travel Chain Cost)

### Koncepcja:

**R5 (Conveyal) oblicza czas podróży jako pełny _Whole Travel Chain_ (WTC):**

Całkowity koszt/czas podróży $c_{ij}$ składa się z:

$$c_{ij} = t_{\text{walk,first}} + t_{\text{wait,1}} + t_{\text{transit,1}} + ... + t_{\text{transfer,k}} + ... + t_{\text{walk,last}}$$

Gdzie:
- **$t_{\text{walk,first}}$** - czas dojścia pieszo od origin do pierwszego przystanku (sieć OSM)
- **$t_{\text{wait,1}}$** - oczekiwanie na pierwszy pojazd (R5 estymuje jako headway/2)
- **$t_{\text{transit,1}}$** - przejazd linią 1 (od przystanku A do przystanku B)
- **$t_{\text{transfer,k}}$** - koszt $k$-tej przesiadki (walk między przystankami + wait na następny pojazd)
- **$t_{\text{walk,last}}$** - czas odejścia pieszo od ostatniego przystanku do destination

### Implementacja w R5:

R5 **automatycznie** oblicza wszystkie składniki WTC:
1. Szuka trasy optymalnej (najkrótszej w czasie) z uwzględnieniem headway'ów i transferów
2. Uwzględnia czasy przesiadek (transfer walk time â‰¤ 200 m domyślnie)
3. Zwraca wynik w kolumnie `travel_time_min` - **to jest już pełny WTC koszt**

### Zaznaczenie: "travel_time_min" z R5 = GTC (Generalized Travel Cost)

Dlatego transformacja w tej komórce jest prosta:
```python
wtc_df = ttm_r5.copy()
wtc_df.rename(columns={'travel_time_min': 'gtc'}, inplace=True)  # GTC = c_ij
wtc_df['gtc_minutes'] = wtc_df['gtc']  # explicite: jednostka to minuty
```

### Opcjonalne rozszerzenia:

W przyszłości można dodać **warianty kosztu** z pewnymi śmiałymi modyfikacjami:

**a) Koszt z "penaltyem za przesiadkę" (jeśli R5 nie uwzględnia go wystarczająco):**
```python
# Jeśli zidentyfikujemy liczbę transferów per para OD
# transfer_penalty = n_transfers * 10  # np. +10 min za każdą przesiadkę
# gtc_with_penalty = gtc + transfer_penalty
```

**b) Koszt z uwzględnieniem niezawodności:**
```python
# Jeśli będziemy mieć dane o zmienności headway'ów
# gtc_with_reliability = gtc + headway_variance_factor
```

**c) Koszt z różnymi tempami percepcji dla grup dochodowych (Etap V):**
```python
# Kalibracja beta per Income group
# gtc_calibrated_by_income = beta_income * gtc + transfer_penalty
```

### Wyjście:

**Macierz WTC zapisywana jako:**
- `outputs/etap3/{city}/wtc_matrix.parquet` - pełna macierz OD x WTC
- `outputs/etap3/{city}/wtc_sample.csv` - sample 1000 par do szybkiego przejrzenia
- `outputs/etap3/{city}/wtc_stats.json` - statystyki (min, max, mean, std, percentyle)

### Statystyki WTC per miasto:

```json
{
  "city": "warsaw",
  "n_pairs": 562500,
  "n_nonnull_pairs": 560488,
  "stats": {
    "min_gtc_minutes": 1.2,
    "max_gtc_minutes": 180,
    "mean_gtc_minutes": 42.7,
    "median_gtc_minutes": 38.5,
    "std_gtc_minutes": 28.3,
    "p50": 38.5,
    "p75": 55.2,
    "p90": 72.1,
    "p95": 85.3
  }
}
```

Te wyjścia (WTC matrices) będą wejściem do **Etapu IV - Modelowanie Grawitacyjne** (calibration decay parameter $\beta$, obliczanie indeksów dostępności).

In [14]:
# --- Dokumentacja impedancji i WTC ---
# R5 travel_time_min jest PEŁNYM Whole Travel Chain (WTC):
#   c_ij = t_walk_first_mile + t_wait + t_in_vehicle + t_transfer + t_walk_last_mile
# Silnik routingowy R5 (Conveyal) optymalizuje trasę uwzględniając wszystkie
# składowe łańcucha podróży, headway-based waiting, przesiadki i marsz po OSM.
# Dlatego transformacja TTM → WTC jest dokumentacyjna (identity), a nie obliczeniowa.

IMPEDANCE_PARAMS = {
    'engine': 'r5py (Conveyal R5)',
    'wtc_components_included': [
        'first_mile_walk (OSM pedestrian network)',
        'waiting_time (headway-based, optimized by R5)',
        'in_vehicle_time (GTFS schedule)',
        'transfer_time (walk + wait at transfer stops)',
        'last_mile_walk (OSM pedestrian network)',
    ],
    'departure_time_window': list(DEPARTURE_TIME_WINDOW),
    'multi_departure_agg': MULTI_DEPARTURE_AGG,
    'walk_speed_kmh': WALK_SPEED_KMH,
    'max_walk_distance_m': MAX_WALK_DISTANCE_M,
    'max_trip_duration_min': MAX_TRIP_DURATION_MIN,
    'cost_variants': {
        'wtc_r5_penalized': {
            'definition': 'cost_min = travel_time_min + est_transfer_penalty (R5 full WTC)',
            'components': 'first_mile + wait + in_vehicle + transfer + last_mile',
            'note': 'R5 travel_time already includes all WTC components; no additional penalties applied',
        },
    },
}

for city in CITIES:
    city_dir = OUT_DIR / city
    wtc_dir = city_dir / 'wtc'
    wtc_dir.mkdir(parents=True, exist_ok=True)
    (wtc_dir / 'impedance_params.json').write_text(json.dumps(IMPEDANCE_PARAMS, ensure_ascii=False, indent=2), encoding='utf-8')

    profile_inputs = []
    for profile_name in TEMPORAL_PROFILES.keys():
        ttm_path = city_dir / 'ttm' / profile_name / f'ttm_raw_{profile_name}.parquet'
        if ttm_path.exists():
            profile_inputs.append((profile_name, ttm_path))

    legacy_ttm_path = city_dir / 'ttm' / 'ttm_raw.parquet'
    if legacy_ttm_path.exists() and not profile_inputs:
        profile_inputs.append(('default', legacy_ttm_path))

    if not profile_inputs:
        raise FileNotFoundError(f'No TTM parquet files found for {city} in {city_dir / "ttm"}')

    first_wtc = None
    for profile_name, ttm_path in profile_inputs:
        ttm = pd.read_parquet(ttm_path)

        wtc = ttm[['origin_id', 'dest_id', 'travel_time_min']].copy()
        if 'is_walk_fill' in ttm.columns:
            wtc['is_walk_fill'] = ttm['is_walk_fill'].astype(bool)
        else:
            wtc['is_walk_fill'] = False
        # WTC: dodajemy psychologiczną karę za przesiadki (Issue 3 fix)
        # R5 nie eksponuje liczby przesiadek na macierzy TTM, stosujemy estymator:
        # ok. 1 przesiadka na każde 20 min jazdy pow. pierwszych 15 minut.
        # Kara = 5 min za przesiadkę, max 3 przesiadki
        t = wtc['travel_time_min']
        est_transfers = ((t - 15) / 20).clip(lower=0).apply(np.floor).clip(upper=3)
        transfer_penalty_min = np.where(wtc['is_walk_fill'], 0.0, est_transfers * np.where(t > 45, 5.0, 10.0))

        wtc['cost_variant'] = 'wtc_r5_penalized'
        wtc['cost_min'] = t + transfer_penalty_min
        wtc['transfer_penalty_applied'] = transfer_penalty_min
        wtc['cost_min_strict'] = wtc['cost_min'].where(~wtc['is_walk_fill'])

        wtc.to_parquet(wtc_dir / f'wtc_matrix_{profile_name}.parquet', index=False)
        if first_wtc is None:
            first_wtc = wtc

    if first_wtc is not None:
        first_wtc.to_parquet(wtc_dir / 'wtc_matrix.parquet', index=False)

print('WTC matrices generated for all cities and profiles')
print(f'Cost variant: wtc_r5_penalized (WTC + estimated psychological transfers)')
print(json.dumps(IMPEDANCE_PARAMS['wtc_components_included'], indent=2))


WTC matrices generated for all cities and profiles
Cost variant: wtc_r5_penalized (WTC + estimated psychological transfers)
[
  "first_mile_walk (OSM pedestrian network)",
  "waiting_time (headway-based, optimized by R5)",
  "in_vehicle_time (GTFS schedule)",
  "transfer_time (walk + wait at transfer stops)",
  "last_mile_walk (OSM pedestrian network)"
]


## 10) Raporty per miasto i zbiorczy summary

### Cel:

Generujemy czytelne **markdown raporty** dla każdego miasta zawierające:
- Liczby i statystyki (ile OD, ile przystanków, jaki engine)
- QC results (czy TTM przeszedł walidację?)
- Ostrzeżenia i rekomendacje (np. "50% to walk-only fallback")
- Metadane uruchomienia (kiedy, jakie parametry)

### Per-city report: `city_name_report.md`

**Struktura:**
```markdown
# Etap III Report - {City}

## 1. Konfiguracja uruchomienia
- **Data analizy:** 2016-06-01
- **Okna czasowe:** poranek (6:00-7:30), południe (11:00-12:30), wieczór (19:00-20:30)
- **Engine:** R5 (lub walk-only)
- **Origins:** 750 (próbka z {total_origins_available})
- **Destinations:** 750 (próbka z {total_destinations_available})

## 2. Walidacja GTFS + OSM
- **Przystanków w Etapie I:** {n_stops_total}
- **Przystanków w obrębie granic adm.:** {n_stops_in_boundary}
- **Przystanków w obrębie 3 km od OD:** {n_stops_within_buffer}
- **GTFS Feed Info:**
  - Agency: {agencies}
  - Routes: {n_routes}
  - Trips: {n_trips} (na tydzień)
  - Stop times: {n_stop_times}

## 3. Zbudowane OD units
- **Origins na siatce 1 km:** {n_origins_total}
- **Destinations na siatce 1 km:** {n_dest_total}
- **Po filtracji do granic adm.:** {n_origins_boundary} / {n_dest_boundary}
- **Po filtracji do 3 km od przystanków:** {n_origins_final} / {n_dest_final}
- **Próbka:** 750 origins, 750 destinations

## 4. Macierz TTM (Travel Time Matrix)
- **Obliczone pary OD:** 750 x 750 = 562,500
- **Niespuste pary:** {n_nonnull} ({share_nonnull:.1%})
- **Engine mode:** {engine}
- **Min travel time:** {min_time} min
- **Max travel time:** {max_time} min
- **Mean travel time:** {mean_time:.1f} min
- **Median travel time:** {median_time:.1f} min
- **Std Dev:** {std_time:.1f} min
- **QC Status:**  PASSED / ⚠️ WARNING / âœ— FAILED

## 5. Warianty WTC (Whole Travel Chain Cost)
- **Base WTC:** travel_time_min z R5 (zawiera walking + waiting + transfers)
- **Fallback:** jeśli {fallback_pct}% to walk-only (bez TZ), flaga LOW_QUALITY

## 6. Pliki wyjściowe
- `od_units/origins.geojson` - {n_origins} origins z atrybutami
- `od_units/destinations.geojson` - {n_dest} destinations
- `ttm_travel_times.parquet` - pełna macierz TTM ({n_pairs} wierszy)
- `wtc_matrix.parquet` - macierz WTC (identyczna, jeśli brak modyfikacji)
- `wtc_stats.json` - statystyki
- `city_config.json` - parametry uruchomienia

## 7. Ostrzeżenia i rekomendacje
- **Jeśli fallback > 30%:** "TTM jakość jest niska - rekomendacja: sprawdzić GTFS/OSM lub uruchomić na maszynie z więcej RAM'u dla R5"
- **Jeśli constant times:** "R5 zwrócił stałe wartości - możliwy crash lub timeout"
- **Jeśli nonnull < 50%:** "Macierz zbyt rzadka - wiele par nieosiągalnych; dialog z R5?"

## 8. Postęp i czas wykonania
- **Czas ładowania danych:** X.Xs
- **Czas R5 (TTM):** X.Xs
- **Czas przetwarzania WTC:** X.Xs
- **TOTAL:** X.Xs
```

### Zbiorczy report: `summary.md`

Agreguje wyniki dla **wszystkich 3 miast razem:**

```markdown
# Etap III - Zbiorczy Summary

## Porównanie miast

| Metric | Dublin | Paris | Warsaw |
|--------|--------|-------|--------|
| Origins (próbka) | 750 | 750 | 750 |
| Destinations | 750 | 750 | 750 |
| TTM Engine | R5 | {engine} | {engine} |
| Niespuste pary | 99.2% | 98.5% | 95.1% |
| Mean travel time | 35.2 min | 42.1 min | 39.7 min |
| Median travel time | 32.5 min | 38.0 min | 36.3 min |

## Status QC
-  Dublin: PASSED
- ⚠️ Paris: PASSED (30% walk-only fill)
-  Warsaw: PASSED

## Następne kroki
- **Etap IV:** Calibration modeli grawitacyjnych (decay parameter Î² per miasto, per income group)
- **Etap V:** Analiza nierówności (spatial equity) - porównanie dostępności między grupami dochodowymi
- **Visualizations:** Heatmapy dostępności, mapy isochron, wykresy CDF
```

### Przeznaczenie raportów:

1. **Audytowalność:** Wyjaśniamy każdy krok, jaki byliśmy w stanie zrobić
2. **Diagnostyka:** Jeśli TTM "wygląda dziwnie", raport chwyci qc flags
3. **Dokumentacja:** Future self/reviewer mogą szybko zrozumieć, co się stało
4. **Publikacja:** Fragmenty raportów pójdą do thesis/papers jako appendix

In [15]:
# Per-miasto reports/etap3_summary.md
for city in CITIES:
    city_dir = OUT_DIR / city

    od_val = json.loads((city_dir / 'qc' / 'validation_report.json').read_text(encoding='utf-8'))
    ttm_qc = pd.read_csv(city_dir / 'ttm' / 'ttm_qc_summary.csv').iloc[0].to_dict()

    # Run info (for departure datetimes)
    run_info_path = city_dir / 'run_info.json'
    city_run = json.loads(run_info_path.read_text(encoding='utf-8')) if run_info_path.exists() else {}

    lines = []
    lines.append(f'# Etap III — raport: {city}')
    lines.append('')
    lines.append(f"Uruchomiono (UTC): `{utc_now_iso()}`")
    lines.append('')

    # Parametry analizy
    lines.append('## Parametry')
    lines.append(f"- od_scale_mode: `{OD_SCALE_MODE}`")
    lines.append(f"- sample_k: `{OD_SAMPLE_K}`")
    lines.append(f"- origin_role: `{od_val['origin_role']}`")
    lines.append(f"- dest_role: `{od_val['dest_role']}`")
    lines.append(f"- engine_mode: `{ttm_qc['engine_mode']}`")
    lines.append(f"- admin_boundary_filter: `{od_val.get('admin_boundary_filter', 'N/A')}`")
    lines.append(f"- stops_proximity_buffer_m: `{od_val.get('stops_proximity_buffer_m', 'N/A')}`")
    lines.append(f"- walk_detour_factor: `{WALK_DETOUR_FACTOR}`")
    lines.append(f"- walk_cap_min: `{FALLBACK_WALK_ONLY_MAX_TIME_MIN}`")
    lines.append('')

    # Departure datetimes (reproducibility)
    dep_dts = city_run.get('departure_datetimes')
    if dep_dts:
        lines.append('## Departure datetimes (reproducibility)')
        for dt_str in dep_dts:
            lines.append(f"- `{dt_str}`")
        lines.append('')

    # Filtracja OD — odczyt z osobnych plików QC (od_filter_origins.json / od_filter_destinations.json)
    filter_o_path = city_dir / 'qc' / 'od_filter_origins.json'
    filter_d_path = city_dir / 'qc' / 'od_filter_destinations.json'
    if filter_o_path.exists() or filter_d_path.exists():
        lines.append('## Filtracja OD (granica administracyjna + bliskość przystanków)')
        for role_label, fpath in [('origins', filter_o_path), ('destinations', filter_d_path)]:
            if fpath.exists():
                rqc = json.loads(fpath.read_text(encoding='utf-8'))
                lines.append(f"### {role_label}")
                lines.append(f"- przed filtrem: {rqc.get('n_input', '?')}")
                lines.append(f"- po granicy admin: {rqc.get('n_after_admin_boundary', '?')}")
                lines.append(f"- po bliskości przystanków: {rqc.get('n_after_stop_proximity', '?')}")
                lines.append(f"- zachowane: {rqc.get('kept_share', '?'):.1%}" if isinstance(rqc.get('kept_share'), (int, float)) else f"- zachowane: {rqc.get('kept_share', '?')}")
        lines.append('')

    # Skala OD
    lines.append('## Skala OD')
    lines.append(f"- n_origins: {od_val['n_origins']}")
    lines.append(f"- n_destinations: {od_val['n_destinations']}")
    lines.append(f"- n_pairs: {od_val['n_pairs']}")
    lines.append('')

    # TTM results
    lines.append('## Wyniki TTM (travel_time_min)')
    lines.append(f"- non-null share: {ttm_qc.get('non_null_rows', 0) / max(ttm_qc.get('matrix_rows', 1), 1):.3f}")
    lines.append(f"- min / median / p90 / max (min): {ttm_qc.get('min_time', 0.0):.1f} / {ttm_qc.get('median_time', 0.0):.1f} / {ttm_qc.get('max_time', 0.0):.1f} / {ttm_qc.get('max_time', 0.0):.1f}")
    lines.append(f"- share capped/max_time: {0.0:.3f}")
    lines.append('')

    # WTC info
    lines.append('## WTC (Whole Travel Chain)')
    lines.append('- wariant kosztu: `wtc_r5_penalized` (R5 WTC + transfer penalty)')
    lines.append('- składowe: first_mile_walk + wait + in_vehicle + transfer + last_mile_walk')
    lines.append('- transformacja: identity (R5 travel_time = WTC)')
    lines.append('')

    # Warnings
    warn_path = city_dir / 'qc' / 'warnings.md'
    if warn_path.exists():
        lines.append('## QC — ostrzeżenia')
        lines.append(warn_path.read_text(encoding='utf-8').strip())
        lines.append('')

    (city_dir / 'reports').mkdir(parents=True, exist_ok=True)
    (city_dir / 'reports' / 'etap3_summary.md').write_text('\n'.join(lines) + '\n', encoding='utf-8')

# Zbiorcze summary.md
summary_lines = []
summary_lines.append('# Etap III — podsumowanie')
summary_lines.append('')
summary_lines.append(f"Uruchomiono (UTC): `{utc_now_iso()}`")
summary_lines.append('')
summary_lines.append('## Parametry')
summary_lines.append(f"- od_scale_mode: `{OD_SCALE_MODE}`")
summary_lines.append(f"- sample_k: `{OD_SAMPLE_K}`")
summary_lines.append(f"- departure_time_window: `{DEPARTURE_TIME_WINDOW}`")
summary_lines.append(f"- multi_departure_agg: `{MULTI_DEPARTURE_AGG}`")
summary_lines.append(f"- admin_boundary_filter: `{USE_ADMIN_BOUNDARY_FILTER}`")
summary_lines.append(f"- stops_proximity_buffer_m: `{STOPS_PROXIMITY_BUFFER_M}`")
summary_lines.append(f"- walk_detour_factor: `{WALK_DETOUR_FACTOR}`")
summary_lines.append(f"- walk_cap_min: `{FALLBACK_WALK_ONLY_MAX_TIME_MIN}`")
summary_lines.append(f"- paris_od_role_mode: `{PARIS_OD_ROLE_MODE}`")
summary_lines.append(f"- cost_variant: `wtc_r5` (R5 full WTC)")
summary_lines.append('')

summary_lines.append('## QC (TTM) — tabela zbiorcza')
summary_lines.append('')


# Load all TTM_QCs
df_list = []
for c in CITIES:
    qc_path = OUT_DIR / c / 'ttm' / 'ttm_qc_summary.csv'
    if qc_path.exists():
        df_list.append(pd.read_csv(qc_path))
if df_list:
    TTM_QC = pd.concat(df_list, ignore_index=True)
else:
    TTM_QC = pd.DataFrame()

# tabelka markdown z TTM_QC
cols = ['city', 'profile', 'engine_mode', 'matrix_rows', 'non_null_rows', 'min_time', 'median_time', 'max_time']
if not TTM_QC.empty:
    md_df = TTM_QC[cols].copy()
else:
    md_df = pd.DataFrame(columns=cols)
summary_lines.append('| ' + ' | '.join(cols) + ' |')
summary_lines.append('| ' + ' | '.join(['---'] * len(cols)) + ' |')
for _, r in md_df.iterrows():
    summary_lines.append('| ' + ' | '.join(str(r[c]) for c in cols) + ' |')
summary_lines.append('')

# QC porównawcze między miastami
summary_lines.append('## QC porównawcze (cross-city)')
summary_lines.append('')

# Sprawdzenie porównywalności parametrów
summary_lines.append('### Porównywalność parametrów')
summary_lines.append(f"- Wspólna siatka odniesienia: grid 1 km × 1 km")
summary_lines.append(f"- Wspólne okno departures: `{DEPARTURE_TIME_WINDOW}`")
summary_lines.append(f"- Wspólne parametry marszu: walk_speed={WALK_SPEED_KMH:.1f} km/h, max_walk={MAX_WALK_DISTANCE_M} m")
summary_lines.append(f"- Wspólny max_trip_duration: {MAX_TRIP_DURATION_MIN} min")
summary_lines.append(f"- Paris OD: `{PARIS_OD_ROLE_MODE}` (grid_to_grid = porównywalne z Dublin/Warsaw)")
summary_lines.append('')

# Ostrzeżenia ogólne
engine_modes = TTM_QC['engine_mode'].unique().tolist()
if len(set(engine_modes)) > 1:
    summary_lines.append('### ⚠ Ostrzeżenia porównawcze')
    summary_lines.append(f"- Różne engine_modes między miastami: {engine_modes}")
    summary_lines.append('- Porównanie wyników między miastami z różnymi engine_modes wymaga ostrożności')
    summary_lines.append('- Wyniki walk_only_fallback są przybliżeniem (euclidean * detour_factor), nie pełnym routingiem')
    summary_lines.append('')

# Median travel time comparison
if len(TTM_QC) >= 2:
    med_col = 'median_time'
    medians = TTM_QC.groupby('city')[med_col].mean()
    summary_lines.append('### Porównanie median czasu podróży')
    for c in medians.index:
        summary_lines.append(f"- {c}: {medians[c]:.1f} min")
    if medians.max() > 0:
        ratio = medians.max() / max(medians.min(), 0.1)
        summary_lines.append(f"- Stosunek max/min mediany: {ratio:.1f}x")
        if ratio > 3.0:
            summary_lines.append(f"- ⚠ Duża dysproporcja median (>{3.0}x) — zweryfikować engine_mode i pokrycie OD")
    summary_lines.append('')

(OUT_DIR / 'summary.md').write_text('\n'.join(summary_lines) + '\n', encoding='utf-8')

print('Summary reports generated')
print('Per-city reports:', [f'{c}/reports/etap3_summary.md' for c in CITIES])
print('Global summary:', OUT_DIR / 'summary.md')

Summary reports generated
Per-city reports: ['warsaw_core/reports/etap3_summary.md', 'dublin_core/reports/etap3_summary.md', 'warsaw/reports/etap3_summary.md', 'paris/reports/etap3_summary.md', 'dublin/reports/etap3_summary.md']
Global summary: C:\Users\Michc\Dropbox\IO_UW\Magisterka\Masters\outputs\etap3\summary.md


## 11) Artifacts index (kontrakt wyjściowy)

### Celowość:

Analogicznie do Etapów I i II, zapisujemy **metafile** `artifacts_index.json`, który dokumentuje:
- Jakie artefakty wygenerowaliśmy
- Gdzie się znajdują (ścieżki względne)
- Podstawowe metadane (format, schema, wersja)

Ta struktura umożliwia:
- **Łańcuchowe etapy:** Etap IV pyta "gdzie jest WTC matrix z Etapu III?" i czyta z artifact index
- **Reproducibility:** Jeśli przeinstalujemy repo, możemy odtworzyć ścieżki
- **Wizualizacja pipeline'u:** Narzędzie może przeanalizować index i rysować DAG (directed acyclic graph) zależności

### Struktura `artifacts_index.json`:

```json
{
  "stage": "etap3",
  "run_utc": "2026-04-24T14:30:00Z",
  "root": "/path/to/repo",
  "out_dir": "/path/to/repo/outputs/etap3",
  "od_units": {
    "warsaw": {
      "origins_geojson": "warsaw/od_units/origins.geojson",
      "destinations_geojson": "warsaw/od_units/destinations.geojson",
      "origins_csv": "warsaw/od_units/origins_lookup.csv",
      "destinations_csv": "warsaw/od_units/destinations_lookup.csv",
      "n_origins": 750,
      "n_destinations": 750
    },
    "dublin": { ... },
    "paris": { ... }
  },
  "ttm": {
    "warsaw": {
      "parquet": "warsaw/ttm_travel_times.parquet",
      "sample_csv": "warsaw/ttm_sample_1k.csv",
      "n_pairs_total": 562500,
      "n_pairs_nonnull": 560123,
      "engine": "r5",
      "analysis_date": "2016-06-01"
    },
    "dublin": { ... },
    "paris": { ... }
  },
  "wtc": {
    "warsaw": {
      "parquet": "warsaw/wtc_matrix.parquet",
      "stats_json": "warsaw/wtc_stats.json",
      "description": "Whole Travel Chain cost matrix (travel_time_min from R5, fully includes walking, waiting, transfers)"
    },
    "dublin": { ... },
    "paris": { ... }
  },
  "reports": {
    "warsaw_report": "warsaw/warsaw_report.md",
    "dublin_report": "dublin/dublin_report.md",
    "paris_report": "paris/paris_report.md",
    "summary": "summary.md"
  },
  "metadata": {
    "od_sample_k": 750,
    "od_random_seed": 42,
    "walk_speed_mps": 1.25,
    "max_walk_distance_m": 2000,
    "fallback_walk_only_max_time_min": 90.0,
    "walk_detour_factor": 1.3
  }
}
```

### Publikacja artifact index:

```python
artifacts_etap3 = {
    'stage': 'etap3',
    'run_utc': utc_now_iso(),
    'root': str(ROOT),
    'out_dir': str(OUT_DIR),
    'od_units': {...},
    'ttm': {...},
    'wtc': {...},
    'reports': {...},
    'metadata': run_info['analysis'],
}

(OUT_DIR / 'artifacts_index.json').write_text(
    json.dumps(artifacts_etap3, ensure_ascii=False, indent=2),
    encoding='utf-8'
)
```

### Integracja z następnymi etapami:

Etap IV (Gravity Models) czyta artifact index:
```python
artifacts_etap3 = json.loads((ROOT / 'outputs/etap3/artifacts_index.json').read_text())
for city in ['warsaw', 'dublin', 'paris']:
    wtc_path = ROOT / 'outputs/etap3' / artifacts_etap3['wtc'][city]['parquet']
    wtc_df = pd.read_parquet(wtc_path)
    # ... dalsze obliczenia
```

Dzięki temu Etap IV jest całkowicie niezależny od struktur katalogów - wszystko jest w pliku JSON.

In [16]:
# artifacts_index.json Etapu III
artifacts_etap3 = {
    'generated_at_utc': utc_now_iso(),
    'od': {},
    'ttm': {},
    'wtc': {},
    'qc': {},
    'reports': {},
}

for city in CITIES:
    city_dir = OUT_DIR / city
    artifacts_etap3['od'][city] = {
        'origins_geojson': str((city_dir / 'od' / 'origins.geojson').resolve()),
        'destinations_geojson': str((city_dir / 'od' / 'destinations.geojson').resolve()),
        'od_units_csv': str((city_dir / 'od' / 'od_units.csv').resolve()),
    }
    ttm_profile_parquets = sorted((city_dir / 'ttm').glob('*/ttm_raw_*.parquet'))
    artifacts_etap3['ttm'][city] = {
        'ttm_raw_parquets': [str(p.resolve()) for p in ttm_profile_parquets],
        'ttm_qc_summary_csv': str((city_dir / 'ttm' / 'ttm_qc_summary.csv').resolve()),
    }
    if ttm_profile_parquets:
        artifacts_etap3['ttm'][city]['ttm_raw_parquet'] = str(ttm_profile_parquets[0].resolve())
    artifacts_etap3['wtc'][city] = {
        'impedance_params_json': str((city_dir / 'wtc' / 'impedance_params.json').resolve()),
        'wtc_matrix_parquet': str((city_dir / 'wtc' / 'wtc_matrix.parquet').resolve()),
    }
    artifacts_etap3['qc'][city] = {
        'validation_report_json': str((city_dir / 'qc' / 'validation_report.json').resolve()),
    }

    # Add warnings only if file exists
    if (city_dir / 'qc' / 'warnings.md').exists():
        artifacts_etap3['qc'][city]['warnings_md'] = str((city_dir / 'qc' / 'warnings.md').resolve())

    artifacts_etap3['reports'][city] = {
        'etap3_summary_md': str((city_dir / 'reports' / 'etap3_summary.md').resolve()),
    }

(OUT_DIR / 'artifacts_index.json').write_text(json.dumps(artifacts_etap3, ensure_ascii=False, indent=2), encoding='utf-8')

print('Wrote:', OUT_DIR / 'summary.md')
print('Wrote:', OUT_DIR / 'artifacts_index.json')


Wrote: C:\Users\Michc\Dropbox\IO_UW\Magisterka\Masters\outputs\etap3\summary.md
Wrote: C:\Users\Michc\Dropbox\IO_UW\Magisterka\Masters\outputs\etap3\artifacts_index.json


## 12) Podsumowanie i następne kroki

### Co zostało ukończone w Etapie III:

**1. Origin-Destination (OD) Units**
-  Załadowaliśmy siatki 1 km z Etapu I dla trzech miast
-  Filtracja do granic administracyjnych (Dublin co., Paryż EPT, Warszawa + powiaty)
-  Filtracja do bliskości przystanków (3 km buffer)
-  Próbkowanie: 750 origins x 750 destinations per miasto
-  Zapis do GeoJSON + lookup CSV

**2. Travel Time Matrix (TTM) za pomocą R5**
-  Konfiguracja R5 per miasto (GTFS + OSM)
-  Obliczenie macierzy czasów podróży (562,500 par OD per miasto)
-  Uwzględnienie pełnego Whole Travel Chain (first-mile walking, waiting, transit, transfers, last-mile walking)
-  Fallback walk-only dla awaryjnych scenariuszy
-  QC walidacja (constant times check, sparse matrix check, engine mode logging)

**3. Whole Travel Chain (WTC) Cost Matrix**
-  R5 macierze czasu **są już** `travel_time_min` = GTC (generalized travel cost $c_{ij}$)
-  Zapis do Parquet (dla wydajności)
-  Statystyki per miasto (min, max, mean, median, percentyle)

**4. Raportowanie i Audytowalność**
-  Per-city reports (.md) z diagnostyką i QC flags
-  Zbiorczy summary comparative statistics między miastami
-  JSON metadata (run_info, inventory) dla replikowalności
-  Artifacts index (kontrakt dla Etapów IV/V)

In [17]:
# Podgląd summary
print((OUT_DIR / 'summary.md').read_text(encoding='utf-8'))


# Etap III â€” podsumowanie

Uruchomiono (UTC): `2026-04-25T18:24:59Z`

## Parametry
- od_scale_mode: `sample_k`
- sample_k: `750`
- departure_time_window: `('06:00:00 - 7:30:00', '11:00:00 - 12:30:00', '19:00:00 - 20:30:00')`
- multi_departure_agg: `median`
- admin_boundary_filter: `True`
- stops_proximity_buffer_m: `3000`
- walk_detour_factor: `1.3`
- walk_cap_min: `90.0`
- paris_od_role_mode: `grid_to_grid`
- cost_variant: `wtc_r5` (R5 full WTC)

## QC (TTM) â€” tabela zbiorcza

| city | profile | engine_mode | matrix_rows | non_null_rows | min_time | median_time | max_time |
| --- | --- | --- | --- | --- | --- | --- | --- |
| warsaw_core | morning_peak | r5py_plus_walk_fill | 265225 | 265225 | 0.0 | 67.0 | 179.0 |
| warsaw_core | midday_offpeak | r5py_plus_walk_fill | 265225 | 265225 | 0.0 | 71.0 | 170.0 |
| warsaw_core | evening | r5py_plus_walk_fill | 265225 | 265225 | 0.0 | 69.0 | 168.0 |
| dublin_core | morning_peak | r5py_plus_walk_fill | 84790 | 84790 | 0.0 | 73.0 | 175.0 |
|